# Sesión 4 — MUBD

## Regresión logística: de la probabilidad a la decisión

**Pregunta de la sesión:** a partir de las características registradas de un crédito, ¿qué probabilidad tiene de pagarse? En la sesión 3 predecíamos una cantidad, como la resistencia del hormigón. Aquí solo hay **dos resultados observados**: paga o no paga.

Conviene separar tres cosas desde el principio:

| Concepto | Qué representa | Ejemplo ficticio |
|---|---|---|
| Resultado real, `y` | Lo que finalmente ocurrió | El crédito se pagó: `yes`, codificado como 1 |
| Probabilidad predicha, `p` | Lo que estima el modelo antes de conocer ese resultado | `p = 0.80`: probabilidad estimada de pago del 80 % |
| Clase predicha | La etiqueta que asigna una regla basada en `p` | Con corte 0,5 y regla `p > 0.5`, se predice pago |

**Una probabilidad no es una certeza.** Si un grupo de créditos comparables tiene probabilidades bien estimadas próximas a 0,80, esperamos que pague aproximadamente el 80 % del grupo; eso no identifica con certeza qué créditos concretos pagarán.

### Cómo recorrer el cuaderno

Seguiremos **describir → codificar → ajustar → seleccionar → diagnosticar → interpretar → evaluar**. En cada apartado: lee la pregunta, ejecuta el ejemplo, identifica la salida y redacta una frase antes de pasar a la micropráctica. Los huecos `...` y los ejercicios **4.1 y 4.2** son para practicar: no hace falta completarlos para ejecutar los ejemplos posteriores. Si aparece «pendiente» o «revisa los huecos» en una celda de ejercicio, todavía no has completado esa actividad.

- **Imprescindible en una primera lectura:** distinguir resultado, probabilidad y decisión; comprender referencias y OR; interpretar calibración, AUC y los denominadores de las métricas.
- **Para profundizar:** las fórmulas de máxima verosimilitud y Hosmer–Lemeshow, el estadístico z y el cálculo de intervalos por perfil. Primero entiende qué pregunta responde cada herramienta; después vuelve a su fórmula.

**Regla de trabajo:** entrenamiento sirve para aprender y seleccionar; test sirve para evaluar el modelo y el umbral ya fijados. No escogemos otro modelo ni otro corte porque mejore el test. El ejemplo es educativo y **no recomienda una política real de concesión de créditos**.

Selecciona el kernel del proyecto y ejecuta en orden con **Shift + Enter**. Los widgets son opcionales y tienen alternativas estáticas. Para repetir todo desde cero: **Reiniciar kernel → Ejecutar todo**. [Guía de notebooks en VS Code](https://code.visualstudio.com/docs/datascience/jupyter-notebooks).

### Variables del conjunto de datos

**Cómo leer la tabla:** cada fila de los datos representa un crédito. Las columnas describen sus características y su resultado. Las **predictoras** son la información de entrada; la **respuesta**, lo que intentamos anticipar. Aquí la respuesta es exclusivamente `y`.

| Variable | Descripción |
| --- | --- |
| `status` | Situación de la cuenta corriente |
| `duration` | Duración del crédito, en meses |
| `credit.hist` | Historial crediticio |
| `purpose` | Finalidad del crédito |
| `credit.amo` | Importe del crédito |
| `savings` | Cuenta de ahorro o bonos |
| `employed.time` | Antigüedad en el empleo actual |
| `installment` | Cuota como porcentaje de la renta disponible |
| `status.sex` | Estado civil y sexo |
| `debtors` | Otros deudores o avalistas |
| `residence.time` | Tiempo en la residencia actual |
| `property` | Propiedades |
| `age` | Edad, en años |
| `installment2` | Otros planes de pago a plazos |
| `housing` | Tipo de vivienda |
| `num.credits` | Número de créditos existentes en este banco |
| `job` | Tipo de empleo |
| `num.people` | Número de personas a cargo |
| `telephone` | Disponibilidad de teléfono |
| `foreign` | Trabajador extranjero |
| `y` | Crédito pagado: `no` o `yes` |

**Dos tipos de entrada:** una variable numérica, como `duration`, permite hablar de diferencias de meses; una categórica, como `purpose`, identifica grupos y no se suma ni se promedia como una cantidad. Los nombres se mantienen en inglés porque así aparecen en el archivo y en el código.

**Atención a `yes`:** `foreign=yes` describe una característica registrada; `y=yes` indica que el crédito se pagó. Son columnas distintas. Ni `y` ni una copia de su resultado deben entrar como predictoras: sería revelar al modelo la respuesta que queremos anticipar.

La presencia de características personales en este conjunto histórico no demuestra que sea lícito, pertinente o justo utilizarlas en una decisión real.

## 4.0. Preparación y lectura de datos

### Bibliotecas y entorno

**Objetivo:** preparar las herramientas antes de leer los créditos. El **kernel** es el proceso de Python que ejecuta las celdas y recuerda sus variables. Debe usar el entorno del proyecto, donde están instaladas las bibliotecas; no hace falta instalarlas en cada ejecución.

| Herramienta | Para qué la usaremos |
|---|---|
| `pandas` (`pd`) | Leer y manejar tablas |
| `NumPy` (`np`) | Operaciones numéricas, como exponenciales |
| `Matplotlib` (`plt`) | Dibujar gráficos |
| `statsmodels` (`sm`) | Estimar el modelo logístico y su incertidumbre |
| `scikit-learn` | Calcular ROC y AUC |

`import` carga una herramienta; `as` le asigna un nombre corto. Por ejemplo, `pd.read_csv(...)` significa «usar la función de lectura de pandas». `display(...)` muestra tablas con formato.

Las funciones largas están separadas para que puedas centrarte en sus **entradas y resultados**, sin programar todo el algoritmo:

- [s4_modelos.py](s4_modelos.py): ajuste, selección, contrastes, intervalos y métricas.
- [s4_graficos.py](s4_graficos.py): gráficos y alternativas estáticas.
- [s4_controles.py](s4_controles.py): controles interactivos opcionales.

**Qué hacen las dos primeras celdas:** localizan la raíz del proyecto, permiten importar esos scripts y cargan las bibliotecas. Las líneas que empiezan por `%` son instrucciones de Jupyter: muestran gráficos dentro del cuaderno y permiten ver varias salidas de una misma celda.

**Una ayuda para leer `partial`:** más adelante fijaremos los datos de una función para no escribirlos en cada llamada. Es como dejar preparada una herramienta con la tabla que va a utilizar; no ajusta un modelo por sí solo ni cambia los datos. No necesitas dominar esa técnica para interpretar el análisis.

**Si algo falla:** `ModuleNotFoundError` suele indicar un entorno incorrecto o una ruta de importación no preparada; `NameError`, una variable todavía no creada. Comprueba el kernel y ejecuta primero las celdas anteriores.

In [ ]:
from pathlib import Path  # Rutas portables al proyecto.
import sys  # Permite importar los scripts de esta sesión.

PROYECTO = next((carpeta for carpeta in (Path.cwd(), *Path.cwd().parents)
                 if (carpeta / 'datos' / 'bank0_train.txt').is_file()), None)
if PROYECTO is None:
    raise FileNotFoundError('Abre el cuaderno dentro del proyecto LaSalle_Estadistica.')
SCRIPTS_PATH = PROYECTO / 'sesiones' / 'sesion4'
if str(SCRIPTS_PATH) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_PATH))  # No depende del directorio inicial del kernel.

In [ ]:
from functools import partial  # Fija los datos de una función sin copiar su implementación.
import numpy as np  # Operaciones numéricas y odds ratios.
import pandas as pd  # Lectura y resumen de tablas.
import matplotlib.pyplot as plt  # Visualización y cierre de figuras incompletas.
import statsmodels.api as sm  # Diseño y modelo binomial.
from IPython.display import display  # Muestra tablas y controles.
from sklearn.metrics import roc_auc_score  # Discriminación, no exactitud.
from s4_graficos import configurar_estilo, mostrar_html  # Utilidades externas al cuaderno.

%matplotlib inline
%config InteractiveShell.ast_node_interactivity = "all"
configurar_estilo()  # Conserva el formato de los gráficos originales.

### 4.0.1. Leer el conjunto de entrenamiento

**Pregunta:** ¿se ha leído correctamente la tabla con la que aprenderá el modelo? Antes de calcular nada necesitamos comprobar sus filas, columnas y posibles ausentes.

La llamada `pd.read_csv(..., sep=';')` abre el archivo y separa sus columnas por **punto y coma**. El resultado se guarda en `datos`, una tabla de pandas. `DATA_PATH` indica dónde está la carpeta de datos; no contiene las observaciones, sino su ubicación.

Lee las operaciones siguientes en este orden:

1. **Leer:** cada fila corresponde a un crédito y la primera fila del archivo proporciona los nombres de las columnas.
2. **Comprobar ausentes:** `isna()` marca las celdas vacías; las dos llamadas a `any()` preguntan si hay alguna en toda la tabla. El código se detiene si encuentra una, para no tratarla como una categoría conocida.
3. **Reconocer categorías:** las columnas de texto pasan a `category`. Esto conserva sus niveles —por ejemplo, los tipos de vivienda—, pero todavía no crea los indicadores que necesitará el modelo.
4. **Inspeccionar:** `shape` devuelve filas y columnas; `head()` enseña las primeras cinco filas; `describe(include='all')` resume también las categorías; `value_counts()` cuenta cada resultado de `y`.

**Importante:** ver cinco filas no significa que solo se hayan leído cinco. `head()` limita la visualización, no recorta `datos`.

La respuesta sigue siendo `y`: **`yes` significa paga y `no`, no paga**. Todavía no hemos aprendido coeficientes ni calculado probabilidades. Convertir una columna a categoría tampoco demuestra que sea útil para predecir.

In [ ]:
DATA_PATH = PROYECTO / 'datos'  # Usa la raíz localizada en la preparación.

datos = pd.read_csv(DATA_PATH / 'bank0_train.txt', sep=';')  # Entrenamiento separado por punto y coma.
if datos.isna().any().any():  # No convierte categorías ausentes en indicadores de referencia.
    raise ValueError('Revisa los valores ausentes antes de codificar el entrenamiento.')
for columna in datos.select_dtypes(include=['object', 'str']):
    datos[columna] = datos[columna].astype('category')  # Conserva sus niveles categóricos.

In [ ]:
print(f'Entrenamiento: {datos.shape[0]} créditos y {datos.shape[1]} variables.')  # Incluye la respuesta y las predictoras.
display(datos.head())  # Permite inspeccionar las primeras filas; datos conserva toda la muestra.

In [ ]:
display(datos.describe(include="all"))  # Resume números y categorías; unique/top/freq describen las categóricas.
display(datos['y'].value_counts().rename('Créditos por resultado'))  # Cuenta pagos e impagos antes de modelizar.

### Lectura de la inspección inicial

**Empieza por el tamaño y por la respuesta; después lee las otras columnas.**

- **687 filas y 21 columnas:** hay 687 créditos, **20 predictoras y una respuesta**. El índice de la izquierda sirve para localizar filas; no es una variable de entrada.
- **473 pagos y 214 impagos:** la proporción de pagos es $473/687\approx68{,}9\%$. Esta proporción describe entrenamiento; no es todavía una probabilidad adaptada a cada crédito.
- **Una referencia sencilla:** decir siempre «paga» acertaría el 68,9 % en estas filas, pero no detectaría ningún impago. Por eso un porcentaje global de aciertos puede ser engañoso.
- **Variables numéricas:** la duración mediana es 18 meses y la edad mediana, 33 años. La mediana deja aproximadamente la mitad de los valores a cada lado. El importe tiene media 3231,67 y mediana 2278; unos importes altos pueden elevar la media.
- **Grupos pequeños:** `foreign` tiene 657 casos `yes` y solo 30 `no`. Una comparación basada en pocos casos suele ser menos precisa; el tamaño no debe ignorarse al leer sus coeficientes.

**Cómo interpretar el resumen de categorías:** `unique` cuenta niveles diferentes, `top` indica el más frecuente y `freq`, cuántas veces aparece. Los `NaN` de una fila como `mean` para una categoría indican que esa medida no se aplica; no equivalen a ausentes en los datos originales. La comprobación de ausentes se hace con `isna()`.

**Qué sabemos ahora:** tamaño, distribución de la respuesta y algunas características de las entradas. **Qué no sabemos:** qué variables predecirán mejor, si las probabilidades estarán bien estimadas o si la muestra representa a una población concreta.

**Comprueba antes de seguir:** explica por qué «el 68,9 % paga» y «este crédito tiene una probabilidad de pago del 68,9 %» no son la misma afirmación.

## 4.1. Exploración bivariante

### 4.1.1. Variables categóricas

**Pregunta:** ¿se distribuyen igual las características de los créditos que pagaron y de los que no pagaron? Antes del modelo, observamos relaciones sencillas entre **una entrada y la respuesta**. A esto lo llamamos exploración bivariante.

`datos.dtypes` informa del tipo de cada columna. `var_cat` reúne los nombres de las columnas categóricas y excluye `y`. La expresión `if columna != 'y'` significa «conserva esta columna solo si no es la respuesta».

**Por qué distinguimos tipos:** no tiene sentido calcular una edad media para las etiquetas de vivienda; sí tiene sentido contar cuántos créditos hay en cada tipo. Para comparar categorías utilizaremos un mosaico y una tabla de frecuencias.

Busca tres cosas: diferencias de composición, categorías con pocos casos y patrones que merezcan estudio. Una frecuencia es un **recuento**; una proporción divide ese recuento por un total que debemos especificar.

**Límite esencial:** estas son relaciones **marginales**, sin controlar las demás entradas. Que dos grupos tengan proporciones de pago diferentes puede estar relacionado con otras diferencias entre ellos. No estamos estimando todavía una asociación ajustada ni un efecto causal.

In [ ]:
datos.dtypes  # Comprueba qué columnas son numéricas y cuáles conservan niveles categóricos.

In [ ]:
var_cat = [columna for columna in datos.select_dtypes(include="category") if columna != "y"]  # Excluye la respuesta.
print(var_cat)  # Lista los nombres que se pueden utilizar en el mosaico.

#### Diagramas de mosaico

**Qué vamos a dibujar:** una comparación de las categorías de `status` —situación de la cuenta corriente— entre los créditos que pagaron y los que no pagaron.

Lee el mosaico en dos pasos:

1. **Anchura de cada bloque de respuesta:** el bloque de `Paga` será más ancho si hay más pagos que impagos en toda la muestra.
2. **Altura de cada categoría dentro de un bloque:** indica qué fracción de ese grupo de respuesta pertenece a la categoría. El área de cada rectángulo combina ambas frecuencias.

**La pregunta del mosaico no debe invertirse sin calcular.** «¿Qué proporción de quienes pagaron tenía esta cuenta?» no es lo mismo que «¿qué proporción de quienes tenían esta cuenta pagó?». Para la segunda pregunta, divide los pagos de la categoría por **todos los créditos de esa categoría**, usando la tabla que aparece debajo.

**Cómo leer las llamadas:** primero se importa la función y se fija `datos` con `partial`; después basta escribir `mosaico_pago('status')`. Para cambiar de entrada, sustituye ese nombre por otro de `var_cat`. El texto entre comillas debe coincidir con una columna existente.

**Antes de ejecutar:** anticipa qué bloque será más ancho a partir de los recuentos de `y`. Después localiza una categoría pequeña: un rectángulo casi invisible puede ser consecuencia de pocos casos, no de una probabilidad de pago exactamente cero.

In [ ]:
from s4_graficos import mosaico_pago as _mosaico_pago

mosaico_pago = partial(_mosaico_pago, datos)  # Fija entrenamiento; después solo se elige la variable.

In [ ]:
mosaico_pago("status")  # Prueba también otros nombres de var_cat.

### Lectura del mosaico de `status`

El bloque de pagos es más ancho porque hubo más pagos que impagos. Dentro de ese bloque, `no checking account` tiene bastante presencia. Esa altura describe la composición de los pagos; **no es directamente la probabilidad de pago de la categoría**.

La tabla permite responder a esa segunda pregunta:

| Categoría | Pagos / total de la categoría | Proporción observada de pago |
|---|---|---|
| `no checking account` | 236 / 269 | 87,7 % |
| `< 0 DM` | 92 / 195 | 47,2 % |

**Cómo se obtiene el denominador:** suma los impagos y los pagos de la misma fila de la tabla. Comparar solo 236 con 92 no basta, porque los grupos tienen tamaños distintos. El 87,7 % es una proporción observada de entrenamiento, no el resultado de una regresión ni una garantía para otro crédito.

**Conclusión adecuada:** en esta muestra se observa una asociación entre situación de la cuenta y pago. Todavía no mantenemos constantes las demás características y no podemos concluir que el tipo de cuenta cause la diferencia.

#### Micropráctica: comparar otra variable categórica

1. Elige un nombre de `var_cat` distinto de `status` y úsalo en `mosaico_pago(...)`.
2. Describe qué categoría tiene más presencia dentro de pagos e impagos.
3. Usa la tabla para calcular la proporción de pago dentro de dos categorías, indicando sus denominadores.
4. Localiza un grupo pequeño y explica por qué su proporción puede cambiar mucho con solo uno o dos casos adicionales.

**Comprobación de tu respuesta:** debe incluir nombre de la entrada, recuentos, proporciones y una cautela. No basta decir que «un rectángulo es más grande».

In [ ]:
try:
    mosaico_pago(...)  # Sustituye por el nombre de otra variable categórica que aparezca en var_cat.
except Exception as error:
    plt.close("all")  # Cierra una figura parcial si el nombre no es válido.
    print("El microejercicio no está bien implementado todavía. Revisa el nombre de la variable y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### 4.1.2. Curvas condicionales para variables numéricas

**Pregunta:** ¿cómo cambia la frecuencia de pago al pasar de edades pequeñas a grandes, o de duraciones cortas a largas? Una variable numérica puede tomar muchos valores; en lugar de crear una categoría por cada valor, dibujamos una tendencia suave.

La función `densidad_pago` estima de forma exploratoria

$$P(\text{paga}\mid x),$$

que se lee «probabilidad de pagar **entre casos con valores cercanos a** $x$». La barra vertical significa «condicionado a», no una división.

**Idea del cálculo, sin necesidad de programarlo:**

1. Separa los casos que pagaron de los que no pagaron.
2. Alrededor de cada valor coloca una campana gaussiana, llamada **kernel**. Los casos cercanos contribuyen más que los lejanos a la estimación local.
3. Suma las contribuciones de cada clase, conservando su frecuencia, y las normaliza. Para cada valor de la rejilla, las probabilidades de pago e impago suman 1.

La rejilla contiene 512 puntos: son posiciones donde dibujar la estimación, **no 512 clientes nuevos**.

**Cómo leer el gráfico apilado:** el eje horizontal es la entrada elegida; la zona granate representa `No paga` y la azul, `Paga`. La probabilidad de pago se lee como el **grosor vertical de la franja azul**, no como la altura de su borde superior, que siempre llega a 1. Si la frontera granate–azul está en 0,30, la franja azul mide 0,70: el pago estimado es del 70 %.

**Qué hace `suavizado`:** un valor pequeño sigue más oscilaciones y ruido; uno grande combina vecindarios más amplios y produce una curva más suave. El ejemplo usa `age` y valor 1.

**Qué no hace esta función:** no ajusta todavía la regresión logística, no controla otras entradas y no demuestra causalidad. En extremos con pocos datos, una curva dibujada con precisión visual puede tener poca información que la sostenga.

In [ ]:
var_num = [columna for columna in datos.select_dtypes(include="number") if columna != 'y']  # Entradas numéricas, sin respuesta.
print(var_num)  # Nombres disponibles para explorar sus curvas condicionales.

In [ ]:
from s4_graficos import densidad_pago as _densidad_pago

densidad_pago = partial(_densidad_pago, datos)  # Conserva la llamada variable/suavizado de los ejemplos.

In [ ]:
densidad_pago("age", suavizado=1.0)  # Cambia la variable o prueba, por ejemplo, 0.5 y 2.0.

### Lectura de la curva de edad

La estimación cambia con la edad y presenta oscilaciones. Eso indica que, en estos datos y con este suavizado, la proporción local de pagos no es constante. **No significa que cada subida o bajada sea una diferencia real y estable de riesgo.**

Las edades observadas van de **19 a 75 años**, pero la rejilla se extiende algo más allá para dibujar las campanas. Las zonas fuera de ese rango y los extremos poco poblados no deben interpretarse como predicciones fiables para nuevas personas.

**Recuerda la lectura de colores:** si aumenta el grosor azul, aumenta la probabilidad estimada de pago; si aumenta el granate, ocurre lo contrario. La suma de ambos grosores es 1 en cada punto.

**Conclusión adecuada:** aparece una asociación marginal que merece estudio. La forma depende del suavizado y del apoyo de datos; aún no sabemos si se mantendrá al considerar simultáneamente las otras entradas.

#### Micropráctica: modificar el suavizado

1. Elige otra entrada de `var_num`, distinta de `age`.
2. Completa `densidad_pago(..., suavizado=...)` con un valor mayor que 1.
3. Para distinguir el efecto de la variable del efecto del suavizado, compara **esa misma entrada** con suavizados 1 y 2, por ejemplo.
4. Describe qué detalles desaparecen o se mantienen y menciona una zona con poco apoyo muestral.

**Qué debes anticipar:** un suavizado mayor suele producir menos oscilaciones, pero no vuelve correcta la curva automáticamente. No compares edad con suavizado 1 y duración con suavizado 2 atribuyendo toda diferencia al suavizado: habrías cambiado dos cosas a la vez.

In [ ]:
try:
    densidad_pago(..., suavizado=...)  # Completa con una variable de var_num distinta de age y un suavizado, por ejemplo 2.0.
except Exception as error:
    plt.close("all")  # Cierra una figura parcial si los argumentos no son válidos.
    print("El microejercicio no está bien implementado todavía. Revisa la variable y el suavizado y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### Exploración opcional con controles

El desplegable elige una entrada numérica y el deslizador cambia el suavizado. Empieza dejando una variable fija y mueve **solo el suavizado**; después conserva un valor de suavizado y cambia la entrada. Así podrás distinguir qué decisión explica cada diferencia.

El control llama a la misma `densidad_pago` del ejemplo. **No modifica la tabla, no entrena una regresión logística y no elige variables para el modelo.** Un suavizado muy bajo puede exagerar detalles de la muestra; uno alto puede ocultarlos.

Si los controles no aparecen, no te bloquees: ejecuta `densidad_pago('age', 2.0)` cambiando los argumentos para obtener una figura estática. La interpretación es la misma.

**Para cerrar la exploración:** escribe una tendencia, una limitación y una pregunta que quieras comprobar con el modelo conjunto. No es necesario encontrar una curva «perfecta» antes de continuar.

In [ ]:
from s4_controles import crear_control_densidad

control_densidad = crear_control_densidad(densidad_pago, var_num)  # Solo explora los datos ya fijados.
display(control_densidad)

## 4.2. Codificar el modelo logístico

### 4.2.1. Categorías de referencia

**Problema que resolvemos:** el modelo necesita números, pero varias entradas son etiquetas. No sería adecuado convertir tres tipos de vivienda en 1, 2 y 3 e imponer que la diferencia entre ellos sea siempre una unidad. Usaremos **indicadores**: columnas que valen 1 si un caso pertenece a una categoría y 0 si no.

Para una entrada con tres categorías, bastan **dos indicadores y una referencia**. Ejemplo ficticio:

| Categoría | Indicador B | Indicador C |
|---|---|---|
| A, referencia | 0 | 0 |
| B | 1 | 0 |
| C | 0 | 1 |

Con intercepto, incluir los tres indicadores daría información redundante: siempre sumarían 1. Por eso `drop_first=True` omite el indicador del primer nivel. **No elimina personas ni borra una categoría de los datos**: la referencia está representada cuando los indicadores de esa entrada son cero.

La celda reordena los niveles para decidir cuál será la referencia. Después, un coeficiente de B se interpretará **frente a A**, manteniendo constantes las demás entradas. Si cambiamos la referencia de un modelo equivalente, cambiarán los coeficientes escritos, pero no sus probabilidades predichas.

**En este cuaderno:** `foreign=yes` se compara con `foreign=no`, que solo tiene 30 casos. Para `status`, la base es `no checking account`. Si comparas coeficientes con otro material, comprueba primero las referencias: diferencias de signo o de valor no implican necesariamente modelos diferentes.

**Comprueba:** explica cómo reconoce el modelo a un crédito de la categoría de referencia sin disponer de una columna específica para ella. La referencia no es una categoría «sin riesgo» ni necesariamente la más numerosa.

In [ ]:
from s4_modelos import REFERENCIAS  # Niveles base originales, conservados en el script.

referencias = REFERENCIAS.copy()  # La copia permite explorar cambios sin modificar el módulo.
for columna, referencia in referencias.items():
    categorias = list(datos[columna].cat.categories)
    datos[columna] = datos[columna].cat.reorder_categories(
        [referencia] + [categoria for categoria in categorias if categoria != referencia])
display(pd.Series(referencias, name='Categoría de referencia').to_frame())

### 4.2.2. Del logit a la probabilidad

**¿Por qué no utilizar directamente una recta?** Una expresión como intercepto + pendiente × edad puede tomar cualquier número, incluidos −2 o 1,4. Esos valores no pueden ser probabilidades, que deben estar entre 0 y 1.

La regresión logística conecta una parte lineal con una probabilidad mediante tres escalas:

| Escala | Expresión | Cómo leerla |
|---|---|---|
| Probabilidad | $p$ | Fracción esperada de pagos en casos comparables |
| Odds | $p/(1-p)$ | Relación entre probabilidad de pago e impago |
| Logit o log-odds | $\log(p/(1-p))$ | Logaritmo natural de los odds; puede ser negativo, cero o positivo |

Por ejemplo, $p=0{,}80$ implica odds $0{,}80/0{,}20=4$: el pago tiene cuatro veces la probabilidad del impago. El logit es $\log(4)\approx1{,}386$. **Odds 4 no significa probabilidad 4 ni 400 %.**

El modelo coloca la **suma ponderada de las entradas** —cada entrada multiplicada por su coeficiente, más el intercepto— en la escala logit:

$$\log\!\left(\frac{p_i}{1-p_i}\right)=\eta_i=\beta_0+\sum_j\beta_jx_{ij},\qquad p_i=\frac{1}{1+e^{-\eta_i}}.$$

**Traducción de los símbolos:** $i$ identifica un crédito; $x_{ij}$, su entrada $j$; $\beta_0$, el intercepto; $\beta_j$, un coeficiente; $\eta_i$, la puntuación lineal. La última fórmula convierte esa puntuación en probabilidad. Para una puntuación finita, la probabilidad queda entre 0 y 1.

**Regla de lectura:** $\eta=0$ da $p=0{,}5$; una puntuación positiva da más de 0,5 y una negativa, menos. Un coeficiente positivo eleva los odds de pagar al aumentar su entrada, manteniendo las otras constantes; el cambio en probabilidad no es constante.

**Error que debemos evitar:** no calculamos el logit de cada `y` observado, que vale 0 o 1. Modelamos el logit de la **probabilidad desconocida** de que `y` sea 1. A continuación veremos ejemplos ficticios antes de aprender los coeficientes reales.

#### De la probabilidad a los odds y de vuelta

**Objetivo:** comprobar con números por qué multiplicar odds no equivale a multiplicar probabilidades. Este ejemplo es ficticio y no utiliza los créditos de entrenamiento.

Partimos de una probabilidad de pago de **20 %** y aplicamos un **odds ratio, OR, igual a 2**:

| Paso | Operación | Resultado |
|---|---|---|
| 1. Pasar a odds | $0{,}20/(1-0{,}20)$ | 0,25 |
| 2. Multiplicar los odds por el OR | $0{,}25\times2$ | 0,50 |
| 3. Volver a probabilidad | $0{,}50/(1+0{,}50)$ | 0,333, aproximadamente |

La probabilidad sube de **20 % a 33,3 %**: unos **13,3 puntos porcentuales**, no hasta el 40 %. Los odds sí se duplican. En general,

$$p_1=\frac{OR\,p_0}{1-p_0+OR\,p_0}.$$

**Cómo usar los controles:** cambia primero el OR manteniendo fija la probabilidad inicial. Después fija OR = 2 y cambia la probabilidad inicial. Verás que el mismo multiplicador de odds produce cambios diferentes en puntos de probabilidad.

En escala logit, multiplicar los odds por 2 equivale a sumar $\log(2)$. No necesitas memorizar la derivación: la idea importante es **coeficiente → multiplicador de odds → cambio de probabilidad dependiente del punto de partida**.

Si los controles no aparecen, utiliza `mostrar_html(comparar_odds(0.2, 2))`. El cálculo es el mismo y no modifica ningún modelo.

**Comprueba:** distingue «13,3 puntos porcentuales» de «13,3 % más». El primero resta dos porcentajes; el segundo requeriría comparar el incremento con el valor inicial.

In [ ]:
from s4_graficos import comparar_odds  # Ejemplo ficticio, no predicciones del modelo.
from s4_controles import crear_control_odds

control_odds = crear_control_odds()  # Conserva los valores iniciales p = 0,2 y OR = 2.
display(control_odds)

#### Forma de la probabilidad logística

**Pregunta:** ¿qué forma aparece al convertir una puntuación lineal en probabilidad? Con una sola entrada $x$,

$$\eta=\beta_0+\beta_1x,\qquad p(x)=\frac{1}{1+e^{-\eta}}.$$

La puntuación $\eta$ sigue una recta, pero la probabilidad suele dibujar una **S**. Se aplana cerca de 0 y de 1 porque no puede seguir aumentando o disminuyendo sin límite.

Las tres curvas siguientes son didácticas: **no se han ajustado a los datos del banco**. Todas tienen intercepto cero para aislar el papel de la pendiente.

- **Pendiente positiva:** al aumentar $x$, aumenta la probabilidad de pago.
- **Pendiente negativa:** al aumentar $x$, disminuye esa probabilidad.
- **Pendiente cero:** la entrada no cambia la probabilidad; con intercepto cero, la línea horizontal está en 0,5.

**Cómo interpretar la intensidad:** una pendiente mayor en valor absoluto hace más rápida la transición, pero el cambio en probabilidad por una unidad de $x$ sigue dependiendo de dónde nos encontremos. No hay un incremento fijo en puntos porcentuales.

**Comprueba después de dibujar:** identifica la curva horizontal, localiza el valor en $x=0$ y explica por qué las curvas no salen de [0, 1]. Los valores de $x$ son ficticios: no representan edades ni importes reales.

In [ ]:
from s4_graficos import curvas_logisticas

curvas_logisticas()  # Muestra pendientes positiva, nula y negativa con intercepto cero.

### Explorar intercepto y pendiente sin ajustar datos

El control siguiente permite mover los dos coeficientes de una curva ficticia. **No usa los datos de crédito ni vuelve a entrenar un modelo.**

Haz estos experimentos de uno en uno:

1. Deja la pendiente en 1 y cambia el **intercepto**. Observa la probabilidad en $x=0$: si el intercepto es 0, es 0,5; si aumenta, esa probabilidad aumenta.
2. Deja el intercepto en 0 y cambia la **pendiente**. Compara signo, dirección y rapidez de la transición.
3. Pon la pendiente en 0. La curva será horizontal; el intercepto decidirá su altura.

La línea horizontal en 0,5 es una referencia para leer el gráfico, no una instrucción para decidir sobre créditos reales.

Si no aparecen controles, ejecuta `mostrar_html(curva_logistica(0, 1))` y cambia sus argumentos. **Conclusión que debes poder explicar:** la parte lineal se escribe en log-odds y después se transforma; un coeficiente no representa directamente una diferencia fija de probabilidad.

In [ ]:
from s4_graficos import curva_logistica  # Alternativa estática sin controles.
from s4_controles import crear_control_logistica

control_curva = crear_control_logistica()  # Intercepto 0 y pendiente 1; no ajusta créditos.
display(control_curva)

## 4.3. Ajustar el modelo completo

### 4.3.1. Máxima verosimilitud

**Pregunta:** ¿cómo aprende el modelo los coeficientes a partir de los datos? Hasta ahora los elegíamos en ejemplos ficticios. Ahora utilizamos las respuestas observadas del entrenamiento para estimarlos.

Cada crédito tiene resultado 0 o 1. Para una probabilidad de pago $p_i$, observar un pago aporta $p_i$ a la probabilidad del resultado y observar un impago aporta $1-p_i$. Por ejemplo, si el resultado fue pago, una probabilidad propuesta de 0,8 hace ese resultado más probable que una de 0,2; si fue impago, sucede lo contrario.

**Máxima verosimilitud** busca los coeficientes que hacen más compatibles todos los resultados observados a la vez. No memoriza un coeficiente diferente para cada cliente: aprende una relación común entre entradas y probabilidades. El software prueba y actualiza los coeficientes hasta converger.

**Cómo leer las siguientes celdas:**

1. `y = (datos['y'] == 'yes').astype(int)` convierte pago en 1 e impago en 0. Aquí «positivo» significa **paga**, no «situación deseable» en cualquier otro problema.
2. `predictores` excluye la respuesta para que no se use como entrada.
3. `pd.get_dummies(..., drop_first=True)` convierte categorías en indicadores numéricos, conservando las referencias elegidas.
4. `sm.add_constant(...)` añade `const`, una columna de unos para estimar el intercepto.
5. `grupos` vincula los indicadores de cada variable original; después se retirarán juntos si seleccionamos esa variable para eliminarla.
6. `sm.GLM(..., family=sm.families.Binomial()).fit()` estima el modelo. La función auxiliar fija el mismo entrenamiento para todas las llamadas.

**Supuestos que importa recordar:** cada respuesta es **Bernoulli**, es decir, vale 1 con probabilidad $p_i$ y 0 con probabilidad $1-p_i$. Suponemos independencia entre observaciones condicionada a las entradas, una forma adecuada del logit y suficiente información en los grupos. No exigimos una respuesta normal. Si una categoría separa completamente pagos e impagos, algunos coeficientes pueden no tener una estimación finita fiable. Convergencia significa que terminó el algoritmo, no que todos los supuestos sean correctos.

<details><summary>Para profundizar: la función que se maximiza</summary>

Con independencia, la verosimilitud multiplica las contribuciones de todos los créditos. Su logaritmo convierte el producto en suma y conserva el máximo:

$$\ell(\beta)=\sum_i\left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right].$$

Cuando $y_i=1$ queda el término $\log(p_i)$; cuando $y_i=0$, $\log(1-p_i)$. No necesitas derivar esta expresión para interpretar las probabilidades, pero explica por qué las respuestas de entrenamiento intervienen en el ajuste.

</details>

In [ ]:
if datos['y'].isna().any() or not datos['y'].isin(['no', 'yes']).all():  # Evita convertir ausentes o categorías inesperadas en impagos.
    raise ValueError('La respuesta debe contener únicamente no o yes, sin ausentes.')
y = (datos["y"] == "yes").astype(int)  # Suceso modelado: pagar = 1.
predictores = [columna for columna in datos.columns if columna != "y"]  # No utiliza la respuesta como entrada.
X = pd.get_dummies(datos[predictores], drop_first=True, prefix_sep="=", dtype=float)  # Indicadores frente a las referencias.
X = sm.add_constant(X, has_constant="add")  # Añade el intercepto.

grupos = {  # Conserva juntos todos los indicadores de cada variable original.
    variable: [columna for columna in X if columna == variable or columna.startswith(variable + "=")]
    for variable in predictores
}

In [ ]:
from s4_modelos import ajustar_modelo as _ajustar_modelo

ajustar_modelo = partial(_ajustar_modelo, y, X, grupos)  # Fija respuesta, diseño y grupos de entrenamiento.

In [ ]:
mod_glm0 = ajustar_modelo(predictores)  # Ajusta las 20 entradas originales, con todos sus indicadores.
print(f'Convergencia: {mod_glm0.converged}; coeficientes con intercepto: {len(mod_glm0.params)}.')
display(mod_glm0.summary())  # Resume coeficientes, errores estándar, contrastes z y ajuste.

### Lectura del modelo completo

El ajuste **converge** con **687 observaciones y 49 coeficientes**, incluido el intercepto. Son 20 entradas originales: siete numéricas y 13 categóricas. Una categórica puede necesitar varios indicadores; por eso 49 coeficientes no significa 49 variables de los datos.

**No intentes interpretar toda la tabla de una vez.** Empieza con la fila `duration` y estas columnas:

| Columna del resumen | Qué pregunta responde |
|---|---|
| `coef` | ¿Qué cambio se estima en log-odds por unidad o frente a la referencia? |
| `std err` | ¿Qué incertidumbre tiene la estimación del coeficiente? |
| `z` y `P>|z|` | ¿Qué evidencia hay contra un coeficiente nulo bajo el modelo? |
| `[0.025, 0.975]` | ¿Qué intervalo aproximado del 95 % se obtiene para el coeficiente? |

`duration` tiene coeficiente aproximado **−0,0274 por mes** en el modelo completo. La frase correcta es: «una duración mayor se asocia con menores log-odds de pago, manteniendo constantes las demás entradas». **No significa 2,74 puntos porcentuales menos de probabilidad por mes.** Más adelante convertiremos coeficientes a OR para facilitar su interpretación.

Para una fila como `status=...`, la comparación es con la categoría omitida de esa entrada. `const` es el intercepto: corresponde a entradas numéricas cero y a todas las categorías de referencia; ese perfil puede no ser un crédito real del rango observado.

**Sobre los p-valores:** un valor pequeño es evidencia contra $H_0:\beta_j=0$, bajo los supuestos; uno grande no demuestra que la entrada sea irrelevante. `0.000` es un redondeo, no una probabilidad exactamente cero. Un coeficiente grande tampoco implica mayor importancia si las unidades o referencias son distintas.

**Qué no debes leer como acierto:** el pseudo-$R^2$ no es el $R^2$ lineal ni el porcentaje de créditos correctamente clasificados. Convergencia y buen ajuste en entrenamiento no confirman calibración ni rendimiento en test.

**Siguiente paso:** ver si podemos simplificar el diseño sin perder demasiado ajuste según AIC.

In [ ]:
# La serie params muestra los coeficientes que el modelo ha estimado.
mod_glm0.params

#### Microejercicio práctico — columnas y parámetros

**Objetivo:** distinguir los nombres de las entradas originales de los coeficientes que necesita su codificación.

La celda anterior muestra `mod_glm0.params`, una serie con un coeficiente por fila. `len(objeto)` cuenta cuántos elementos tiene un objeto. En cambio, `predictores` contiene los nombres de las entradas originales, antes de crear indicadores.

Completa las dos llamadas a `len(...)` y compara los resultados. Después explica dos motivos por los que no tienen que coincidir:

- El modelo incluye un **intercepto**.
- Una entrada categórica con varios niveles puede aportar **varios coeficientes** frente a su referencia.

**Comprobación de tu interpretación:** no digas que se han inventado más características ni que los indicadores son clientes adicionales. Cambia la representación de las columnas, no el número de créditos.

In [ ]:
try:
    n_coeficientes = len(...)  # Completa con el atributo de mod_glm0 que contiene los coeficientes estimados.
    n_variables_originales = len(...)  # Completa con la lista de predictores del conjunto original.
    print(f"Predictores originales: {n_variables_originales}; coeficientes (incluido intercepto): {n_coeficientes}")
except Exception as error:
    print("El microejercicio no está bien implementado todavía. Revisa los objetos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## 4.4. Selección por AIC

### 4.4.1. Mantener las mismas observaciones

**Pregunta:** ¿necesitamos todas las entradas para describir estos datos? Añadir columnas puede mejorar el ajuste de entrenamiento, pero también aumenta la complejidad y la posibilidad de aprender particularidades de la muestra. Queremos equilibrar ambas cosas.

El **AIC**, criterio de información de Akaike, combina ajuste y número de parámetros:

$$AIC=-2\widehat\ell+2k.$$

Aquí $\widehat\ell$ es la log-verosimilitud del modelo ajustado y $k$, el número de coeficientes estimados, incluido el intercepto. **Para modelos comparables, un AIC menor es preferible.** No es un porcentaje ni una nota de 0 a 10.

**Ejemplo ficticio:** si dos modelos sobre los mismos resultados tienen AIC 710 y 700, el segundo obtiene mejor balance según ese criterio. No significa que cometa un 10 % menos de errores ni que vaya a ganar necesariamente en test.

**Por qué repetimos «las mismas filas»:** si un modelo utiliza 687 créditos y otro pierde filas por ausentes, sus AIC ya no aíslan solo un cambio de especificación. La diferencia podría deberse a haber cambiado la muestra. Por eso comprobamos ausentes y usamos `missing='raise'`, que evita descartarlos sin avisar.

La celda `datos.isna().sum()` devuelve el número de ausentes por columna. **Todos ceros** significa que esta comprobación no encuentra ausentes; no prueba que los datos estén libres de errores ni que sean representativos.

**Qué debes recordar:** AIC compara especificaciones sobre la misma respuesta y las mismas observaciones; no evalúa por sí solo calibración, causalidad o utilidad práctica.

In [ ]:
datos.isna().sum()

### 4.4.2. Eliminación hacia atrás

**Idea intuitiva:** comenzamos con todas las entradas y preguntamos repetidamente «¿podemos quitar una variable completa y obtener menor AIC?». Esta estrategia se llama **backward** o eliminación hacia atrás.

El procedimiento hace lo siguiente:

1. Ajusta el modelo completo.
2. Prueba candidatos quitando una variable original cada vez.
3. Si la variable es categórica, retira **todos sus indicadores juntos**, no solo el que tenga mayor p-valor.
4. Acepta la retirada con menor AIC únicamente si mejora el AIC actual.
5. Repite desde el modelo simplificado; se detiene cuando ninguna retirada lo reduce.

**Cómo leer la salida:** `mod_glm1` guarda el modelo final de este procedimiento; `variables_seleccionadas`, los nombres originales conservados; `pasos_aic`, la historia de retiradas. En la tabla, el paso 0 es el modelo inicial: el guion no es una variable eliminada.

No se eliminan filas ni se consulta test. El intercepto permanece en los candidatos. Como las comparaciones usan el mismo entrenamiento, podemos atribuir sus diferencias de AIC a cambios en las entradas.

**Límite:** es una búsqueda local, no una prueba de todas las combinaciones posibles. Puede ser inestable ante otras muestras. Existen alternativas como forward —añadir entradas— o stepwise —combinar adiciones y retiradas—, pero no las aplicamos aquí.

**No confundir:** selección por AIC no equivale a eliminar todo lo que tenga p-valor mayor que 0,05. Además, los p-valores del modelo elegido no incorporan toda la incertidumbre de haberlo seleccionado con esos mismos datos.

In [ ]:
from s4_modelos import seleccion_atras as _seleccion_atras

seleccion_atras = partial(_seleccion_atras, y, X, grupos)  # Todos los candidatos usan el mismo entrenamiento.

In [ ]:
mod_glm1, variables_seleccionadas, pasos_aic = seleccion_atras(predictores)  # Usa únicamente entrenamiento.
display(pasos_aic)  # Revisa cada retirada, el AIC y el número de variables originales restantes.

In [ ]:
print('Variables originales conservadas:', variables_seleccionadas)  # Una variable categórica puede aportar varios indicadores.
display(mod_glm1.summary())  # El resumen describe el modelo después de seleccionarlo, no una prueba independiente de la selección.

#### Microejercicio práctico — medir el efecto de la selección

**Objetivo:** resumir cuánto se ha simplificado el modelo y cómo ha cambiado el criterio usado para decidirlo.

Reutiliza los objetos anteriores, sin ajustar nada nuevo:

1. Recupera los AIC con `mod_glm0.aic` y `mod_glm1.aic`.
2. Calcula **AIC inicial menos AIC seleccionado**. Con esta convención, un resultado positivo representa una reducción.
3. Cuenta las entradas originales con `len(predictores)` y `len(variables_seleccionadas)`; resta seleccionado a inicial para obtener el número retirado.
4. Comprueba tus resultados en la primera y la última fila de `pasos_aic`.

**Conclusión que debes escribir:** «Se retiraron ... variables originales y el AIC disminuyó en ... sobre los mismos créditos de entrenamiento». Añade por qué esto no demuestra una mejora en test.

No confundas contar variables con contar `params`: una categórica retirada puede hacer desaparecer varios coeficientes a la vez.

In [ ]:
try:
    reduccion_aic = ...  # Resta el AIC de mod_glm1 al AIC de mod_glm0.
    variables_retiradas = ...  # Compara la cantidad de predictores iniciales y seleccionados.
    print(f"Reducción de AIC: {reduccion_aic:.2f}")
    print(f"Variables retiradas: {variables_retiradas}")
except Exception as error:
    plt.close("all")
    print("El microejercicio no está bien implementado todavía. Revisa las operaciones y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### Lectura de la selección

El procedimiento retira primero **`property`**, después `job` y otros siete términos. Pasa de **20 a 11 variables originales** y el AIC baja de aproximadamente **702,65 a 680,05**.

**Cómo interpretar el cambio:** la simplificación pierde algo de ajuste, pero ahorra suficientes parámetros para que AIC la prefiera. El resultado depende del criterio, de los candidatos y de esta muestra de 687 créditos. Una entrada retirada no queda demostrada como inútil en cualquier contexto.

Los 11 nombres seleccionados no son 11 coeficientes: una variable categórica sigue representada por varios indicadores. Tampoco es necesario que todos los indicadores conservados tengan p-valores menores que 0,05, porque la selección evalúa grupos completos con AIC.

**Conclusión adecuada:** «En entrenamiento, el balance ajuste–complejidad favorece una especificación con menos entradas». **Conclusiones que no están justificadas todavía:** «el modelo está perfectamente calibrado», «es justo para todos los grupos» o «predice mejor fuera de la muestra».

A partir de aquí revisaremos sus probabilidades y sus asociaciones. Aunque el modelo se haya simplificado, la evaluación independiente continúa pendiente.

### 4.4.3. Evidencia estadística de los coeficientes

**Pregunta:** ¿qué coeficientes se distinguen con mayor claridad de cero bajo el modelo seleccionado? La tabla siguiente organiza esa evidencia; **no ordena automáticamente la importancia de las variables**.

El estadístico aproximado es

$$z=\frac{\widehat\beta}{SE(\widehat\beta)}.$$

Compara el tamaño del coeficiente con su error estándar. Un mismo coeficiente puede ofrecer poca evidencia si se estima con mucha incertidumbre. Tomamos `abs()` para ordenar por magnitud: tanto valores muy positivos como muy negativos pueden aportar evidencia contra cero.

**Cómo leer el código:** `tvalues` es el nombre del atributo de statsmodels, pero en este GLM binomial sus valores se interpretan como estadísticos **z**; `drop('const')` excluye el intercepto; `sort_values(..., ascending=False)` coloca las mayores magnitudes arriba.

Cada fila corresponde a una columna del diseño, como un indicador de categoría. No es una prueba conjunta de todos los indicadores de su variable original. La columna `p-valor` resume la evidencia frente a un coeficiente nulo bajo los supuestos y la especificación.

**Qué no se puede deducir del orden:** ni mayor efecto en probabilidad, ni mayor mejora en predicción, ni causalidad. El nombre `importancia` se conserva en el código, pero su contenido es una tabla de **evidencia estadística posterior a la selección**.

**Para una primera lectura:** identifica qué mide la tabla y evita llamarla «ranking de predictores más útiles». No es necesario memorizar el cálculo de z para continuar.

In [ ]:
importancia = pd.DataFrame({  # Nombre conservado; la tabla mide evidencia, no importancia predictiva.
    "Evidencia (|z|)": mod_glm1.tvalues.drop("const").abs(),  # En GLM binomial estos son estadísticos z.
    "p-valor": mod_glm1.pvalues.drop("const"),
}).sort_values("Evidencia (|z|)", ascending=False)
display(importancia)  # Indicadores ordenados; las categorías de una variable no forman una prueba conjunta.

## 4.5. Diagnóstico de probabilidades en entrenamiento

### 4.5.1. Pagos observados y esperados

**Pregunta:** cuando el modelo asigna ciertas probabilidades, ¿se parecen a las frecuencias de pago que realmente observamos? Para empezar, agrupamos créditos con puntuaciones parecidas y comparamos sus totales.

`pr = mod_glm1.predict(...)` calcula una probabilidad de pago por fila de **entrenamiento**. Después, `tabla_por_cuantiles(y, pr)` ordena esas probabilidades y forma grupos con números de casos aproximadamente iguales, llamados **cuantiles**. Con diez grupos hablamos de deciles.

| Columna | Cómo se calcula | Qué significa |
|---|---|---|
| `n` | Número de filas del grupo | Créditos comparados |
| `obs` | Suma de respuestas 0/1 | Pagos que ocurrieron |
| `exp` | Suma de probabilidades | Pagos que el modelo espera en conjunto |

**Ejemplo ficticio:** tres créditos con probabilidades 0,2, 0,7 y 0,8 suman **1,7 pagos esperados**. Si dos se pagaron, hay **2 pagos observados**. El 1,7 no es un crédito parcialmente pagado: es una expectativa para el grupo.

La primera fila reúne probabilidades menores y la última, mayores. Los límites de la etiqueta del intervalo están en escala de probabilidad, no son edades ni importes. Si hay muchos empates puede haber menos grupos efectivos.

En el gráfico, compara dentro de cada grupo las barras `obs` y `exp`. Si el modelo espera demasiados pagos, la barra esperada quedará por encima de la observada. Una diferencia pequeña es compatible con variación muestral; no esperamos igualdad exacta en cada grupo.

**Cautela:** estas respuestas ya intervinieron en el ajuste. El diagnóstico de entrenamiento ayuda a detectar problemas, pero puede ser optimista y no sustituye la evaluación con casos nuevos.

In [ ]:
from s4_modelos import tabla_por_cuantiles

pr = mod_glm1.predict(X[mod_glm1.model.exog_names])  # Probabilidades de entrenamiento.
validacion = tabla_por_cuantiles(y, pr)  # Tamaño, pagos observados y esperados por cuantil.
display(validacion)

In [ ]:
from s4_graficos import pagos_por_intervalo as _pagos_por_intervalo

pagos_por_intervalo = partial(_pagos_por_intervalo, y, pr)  # Solo se modifica el número de grupos.

In [ ]:
pagos_por_intervalo(10)  # Cambia el número de grupos y vuelve a ejecutar.

### Lectura de pagos observados y esperados

Se forman diez grupos de **68 o 69 créditos**. Eso confirma que los cuantiles intentan equilibrar tamaños, no dividir el eje de probabilidad en tramos iguales.

- **Primer grupo:** 11 pagos observados frente a aproximadamente 12,93 esperados. El modelo espera algo más de pago del que ocurrió en ese grupo.
- **Último grupo:** 68 observados frente a 67,95 esperados. Sus totales son muy parecidos.

**Cómo redactarlo:** «En el primer grupo, el total esperado supera al observado en unos dos pagos». Es más preciso que «el modelo falla un 2 %», porque aquí hablamos de recuentos y no hemos dividido por ningún total.

Ni una coincidencia casi exacta ni una diferencia concreta validan una probabilidad individual. Los créditos del grupo tienen probabilidades parecidas, no necesariamente idénticas; lo que comparamos es la suma del conjunto.

**Comprueba:** distingue el número de créditos de un grupo (`n`) del número de pagos observados (`obs`). Si un grupo tiene 69 créditos, no significa que deba tener 69 pagos.

El siguiente contraste resume discrepancias de todos los grupos. Es una ayuda adicional, no un sustituto de leer las barras ni una garantía de funcionamiento en test.

### 4.5.2. Contraste de Hosmer–Lemeshow

**Idea esencial:** convertir las diferencias entre pagos observados y esperados por grupos en un estadístico. Preguntamos si esas discrepancias son grandes en relación con lo que esperaríamos bajo la adecuación del modelo por esos grupos.

- **Hipótesis nula, $H_0$:** adecuación del ajuste en la comparación agrupada.
- **Alternativa, $H_1$:** existen discrepancias incompatibles con esa adecuación.
- **p-valor pequeño:** aporta evidencia de falta de ajuste.
- **p-valor grande:** no encontramos evidencia suficiente con este contraste; **no prueba que el modelo sea correcto**.

Bajo la hipótesis nula y la aproximación usada, el p-valor es la probabilidad de obtener un estadístico igual o mayor que el observado. No es la probabilidad de que $H_0$ sea verdadera ni el porcentaje de predicciones correctas.

**Dos maneras de agrupar que no debes confundir:**

| Agrupación | Qué intenta igualar | Dónde la usamos |
|---|---|---|
| Por cuantiles (`qcut`) | Número de créditos por grupo | Hosmer–Lemeshow |
| Por anchura fija (`cut`) | Tramos del eje: 0–0,1; 0,1–0,2; etc. | Gráfico de calibración posterior |

La tabla siguiente compara sus tamaños. Su fila 1 indica «el primer grupo» de cada método, **no necesariamente el mismo intervalo de probabilidades**.

<details><summary>Para profundizar: cómo se calcula el estadístico</summary>

Si $O_g$ y $E_g$ son pagos observados y esperados de un grupo de $n_g$ créditos, sus impagos son $n_g-O_g$ y $n_g-E_g$:

$$H=\sum_g\left[\frac{(O_g-E_g)^2}{E_g}+\frac{((n_g-O_g)-(n_g-E_g))^2}{n_g-E_g}\right].$$

Cada término crece con las discrepancias. Se compara con una distribución $\chi^2$ aproximada de **grupos efectivos menos 2** grados de libertad. Los grados de libertad son un parámetro que determina qué distribución de referencia utilizamos; no son el número de clientes ni un porcentaje. Aquí diez grupos efectivos dan ocho grados de libertad. Esperados muy pequeños debilitan esa aproximación.

</details>

**Límite práctico:** los cortes cambian el resultado y el contraste usa entrenamiento. Léelo junto con los tamaños, las discrepancias y el gráfico; no como un certificado de calibración.

In [ ]:
from s4_modelos import hosmer_lemeshow  # Contraste por grupos efectivos; no prueba calibración perfecta.

In [ ]:
hosmer_lemeshow(y, pr)

# Comparación de tamaños: cada fila representa el orden del grupo, NO el mismo intervalo de probabilidad.
pd.DataFrame({"n (cuantiles)": validacion["n"].reset_index(drop=True).reindex(range(10)).to_numpy(),
              "n (anchura 0,1)": pd.cut(pr, bins=np.linspace(0, 1, 11),
                                        include_lowest=True).value_counts().sort_index().to_numpy()},
             index=pd.Index(range(1, 11), name="Grupo (de menor a mayor)"))

### Lectura de Hosmer–Lemeshow

Con diez grupos efectivos obtenemos aproximadamente **$H=5{,}02$**, **8 grados de libertad** y **$p=0{,}756$**. Si utilizamos como referencia un nivel de significación de 0,05, el p-valor es mayor y **no rechazamos la adecuación en esta comparación agrupada**.

La conclusión no es «hay un 75,6 % de probabilidad de que el modelo sea bueno». Tampoco significa que todas sus probabilidades sean correctas: la agrupación puede ocultar diferencias dentro de cada grupo y el contraste tiene una capacidad limitada para detectarlas.

En la tabla de tamaños, los intervalos de anchura 0,1 contienen desde **9 hasta 191 créditos**, mientras los cuantiles contienen 68–69. Eso explica por qué las dos agrupaciones no tienen por qué dar la misma impresión visual.

#### Micropráctica: sensibilidad a la agrupación

1. Completa `hosmer_lemeshow(y, pr, grupos=...)` para usar **cinco grupos**.
2. Compara estadístico, grados de libertad y p-valor con el ejemplo de diez grupos.
3. Explica qué ha cambiado y qué permanece fijo: cambian los cortes, **no los coeficientes, las probabilidades ni las respuestas**.
4. Redacta la conclusión de cada contraste sin afirmar calibración perfecta.

**Comprobación:** con cinco grupos efectivos, los grados de libertad deben ser tres. No elijas el número de grupos solo para obtener el p-valor que más te guste.

In [ ]:
try:
    resultado_hl_5 = hosmer_lemeshow(y, pr, grupos=...)  # Completa con cinco grupos.
    display(resultado_hl_5)
except Exception as error:
    print("El microejercicio no está bien implementado todavía. Revisa el número de grupos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## 4.6. Interpretar odds ratios e incertidumbre

### 4.6.1. Comparar categorías

**Pregunta:** ¿cómo expresar un coeficiente de forma más comprensible que en log-odds? Lo exponenciamos para obtener un **odds ratio**, OR:

$$OR=e^\beta.$$

`np.exp(...)` calcula esa exponencial. El OR es un **multiplicador de odds** al cambiar una entrada, manteniendo constantes las demás. No tiene unidades de euros ni de probabilidad.

| Valor del OR | Lectura, con el evento pagar |
|---|---|
| Mayor que 1 | Los odds de pago aumentan |
| Igual a 1 | No cambian los odds en esa comparación |
| Entre 0 y 1 | Los odds de pago disminuyen |

**Ejemplo ficticio sin ajustar:** 40 ventas y 20 no ventas por web dan odds 2; 20 ventas y 40 no ventas por correo dan odds 0,5. El OR web/correo es 4, pero las probabilidades son 2/3 y 1/3: no se cuadruplican. En una regresión, además de especificar la comparación, mantenemos las demás entradas constantes.

En la celda siguiente, `mod_glm1.params['foreign=yes']` recupera el coeficiente de ese indicador y `np.exp(...)` lo convierte a OR. La referencia es **`foreign=no`** y el evento modelado es **pagar (`y=yes`)**. No confundas el `yes` de una característica con el `yes` de la respuesta.

**Plantilla de interpretación:** «En este modelo, los odds de pagar de la categoría ... se multiplican por ..., respecto a ..., manteniendo constantes las demás entradas». Acompaña siempre esa frase de su incertidumbre.

Se trata de una asociación histórica condicionada por el modelo: **no es causal ni justifica conceder o denegar créditos por pertenecer a un grupo**. La referencia tiene pocos casos, lo que exige especial cautela.

In [ ]:
try:
    odds_ratio_extranjero = np.exp(mod_glm1.params["foreign=yes"])
    print(f"Odds ratio de foreign=yes frente a foreign=no: {odds_ratio_extranjero:.3f}")
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa el modelo y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### 4.6.2. Comparar una entrada numérica: edad

Con una entrada numérica no comparamos etiquetas, sino **una diferencia de unidades**. `age` está medida en años, de modo que `np.exp(mod_glm1.params['age'])` expresa el OR por **un año adicional**.

Para una separación de $k$ años:

$$OR_k=e^{k\beta_{age}}=\left(e^{\beta_{age}}\right)^k.$$

**Cómo pensar la fórmula:** el coeficiente describe el cambio de log-odds por año. Para varios años, primero multiplicamos ese coeficiente por la diferencia; después exponenciamos. No multiplicamos simplemente el OR anual por el número de años.

La comparación mantiene constantes las otras entradas. Además, supone que la relación de edad con el **logit** es lineal en el rango estudiado. Esto no exige que la probabilidad dibuje una recta.

**Ejemplo de unidades:** un OR anual de 1,02 indica odds aproximadamente un 2 % mayores por año, no dos puntos porcentuales más de probabilidad. El cambio de probabilidad depende de la probabilidad de partida.

**Límite:** no extrapoles a edades fuera del rango observado, 19–75 años. Una fórmula que puede calcularse para cualquier número no garantiza que el modelo describa ese contexto. Los perfiles comparados también deben tener sentido junto con las otras entradas.

In [ ]:
try:
    odds_ratio_edad = np.exp(mod_glm1.params["age"])
    print(f"Odds ratio por un año adicional de edad: {odds_ratio_edad:.3f}")
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa el modelo y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

#### Micropráctica: comparar diez años

**Objetivo:** adaptar una asociación expresada «por año» a una diferencia de edad concreta, sin cambiar el modelo.

1. Recupera el coeficiente de `age` en `mod_glm1.params`.
2. Multiplica el coeficiente por la diferencia solicitada: **diez años**.
3. Aplica `np.exp(...)` al producto y compara el OR con el anual.
4. Interpreta el resultado para dos edades dentro del rango observado, por ejemplo 30 y 40 años, manteniendo constantes las demás entradas.

**Antes de ejecutar:** revisa el signo del coeficiente. Si es positivo, el OR de diez años será mayor que 1 y que el OR anual. Si fuera negativo, quedaría por debajo de 1 y sería menor que el anual.

**Errores que debes evitar:** multiplicar el OR anual por diez; leer el OR como una probabilidad; o interpretar el resultado como un aumento fijo en puntos porcentuales. La asociación tampoco demuestra un efecto causal de envejecer sobre el pago.

In [ ]:
try:
    odds_ratio_edad_10 = np.exp(... * mod_glm1.params["age"])  # Reutiliza la expresión del OR de un año; completa el multiplicador.
    print(f"OR para diez años adicionales: {odds_ratio_edad_10:.3f}")
except Exception as error:
    print("El microejercicio no está bien implementado todavía. Revisa el cálculo y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### Lectura de los odds ratios

**Lee siempre cuatro elementos juntos: evento, comparación, magnitud y condiciones.**

- **`foreign=yes` frente a `foreign=no`: OR ≈ 0,173.** En el modelo seleccionado, los odds de pagar de la primera categoría son aproximadamente 0,173 veces los de la referencia, **manteniendo constantes las demás entradas**. No es una probabilidad de pago del 17,3 % ni una reducción de 82,7 puntos porcentuales.
- **Un año adicional de `age`: OR ≈ 1,020.** En este modelo, los odds de pagar se multiplican aproximadamente por 1,02 por año adicional, **manteniendo constantes las demás entradas**. Esto equivale a unos odds un 2 % mayores, no a dos puntos adicionales de probabilidad.

**Por qué no podemos deducir una probabilidad solo con el OR:** necesitamos también el punto de partida. Para un OR de 2, pasar de una probabilidad inicial de 0,2 da 0,333, mientras partir de 0,5 da 0,667. El multiplicador es el mismo y los incrementos absolutos son distintos.

Las cifras puntuales no cuentan toda la historia: la comparación de `foreign` usa una referencia pequeña y la edad presenta un OR cercano a 1. Los intervalos siguientes permitirán valorar su precisión.

**Comprueba:** redacta una frase de cada ejemplo sin utilizar «causa» ni «probabilidad multiplicada». Estas asociaciones no autorizan decisiones crediticias basadas en pertenencia a grupos.

### 4.6.3. Intervalos de confianza por perfil

**Idea esencial:** el coeficiente calculado con una muestra no es un número conocido con certeza. Un **intervalo de confianza, IC**, resume la precisión de su estimación bajo un procedimiento y unos supuestos.

Un intervalo amplio indica que los datos permiten un rango considerable de asociaciones. Uno estrecho describe mayor precisión, pero no garantiza ausencia de sesgo, causalidad ni importancia práctica. En la interpretación frecuentista, 95 % se refiere a la cobertura del procedimiento al repetir el muestreo; no a una probabilidad del 95 % para un parámetro fijo después de observar estos datos.

**Cómo leer las siguientes operaciones:**

1. `intervalos_perfil(mod_glm1)` calcula límites para los **coeficientes en log-odds** y los guarda en `IC`.
2. `np.exp(...)` transforma ambos límites a escala de **OR**. Como la exponencial es creciente, conserva el orden de los extremos.
3. `drop(index='const')` excluye el intercepto: su exponencial sería el odds del perfil base, no un OR de una entrada.
4. La columna `OR` añade la estimación puntual para verla junto a sus límites.

**Valores de referencia:** coeficiente 0 equivale a OR = 1. Para un OR, busca si el intervalo incluye **1**, no si incluye 0. No incluir 1 ofrece evidencia contra una asociación nula bajo este modelo y procedimiento; incluirlo no demuestra equivalencia ni ausencia de asociación.

**No es un intervalo de probabilidad individual.** No delimita cuánto «pagará» una persona ni un porcentaje de clientes.

<details><summary>Para profundizar: por qué se llama «perfil» y en qué se diferencia de Wald</summary>

El IC de Wald aproxima la incertidumbre mediante $\widehat\beta\pm1{,}96\,SE$ y después exponencia los límites para obtener un IC del OR. Aquí usamos otro procedimiento: fijamos un coeficiente, reajustamos los demás y comprobamos cuánto empeora la log-verosimilitud. Los valores aceptados cumplen aproximadamente

$$2\left[\ell(\widehat\beta)-\ell(\beta_j\text{ fijado})\right]\le\chi^2_{0{,}95;1}.$$

Esos reajustes forman el perfil. Los límites no tienen por qué ser simétricos alrededor de la estimación. La función admite otra confianza mediante `nivel`. No necesitas programar esta búsqueda para interpretar su tabla.

</details>

**Cautela compartida:** los intervalos dependen del modelo y no incorporan toda la incertidumbre de haber seleccionado sus variables.

In [ ]:
from s4_modelos import intervalos_perfil  # Recibe el modelo y devuelve los límites de sus coeficientes.

In [ ]:
IC = intervalos_perfil(mod_glm1)  # Mantiene los límites de coeficientes para los ejercicios.
IC_OR = np.exp(IC.drop(index='const'))  # Exponencia los límites de las predictoras, no el intercepto.
IC_OR.insert(0, 'OR', np.exp(mod_glm1.params.drop(index='const')))  # Añade la estimación puntual.
display(IC_OR.round(3))  # OR y su IC del 95 %; el valor de referencia es 1.

### Lectura de los intervalos de OR

**Empieza por la estimación y después mira ambos extremos.** Para `foreign=yes` frente a `foreign=no`, el OR es aproximadamente 0,173 y su IC del 95 % por perfil, **[0,026; 0,670]**.

- El intervalo queda por debajo de 1: bajo este modelo y procedimiento aporta evidencia contra un OR nulo de asociación, cuyo valor es 1.
- Es amplio: admite multiplicadores de odds bastante distintos. No conviene presentar 0,173 como una magnitud precisa sin más.
- La referencia solo tiene 30 casos y hubo selección previa: ambas circunstancias importan al valorar el resultado.

**Frase incorrecta:** «La probabilidad de pago está entre 2,6 % y 67 %». **Frase adecuada:** «El intervalo se refiere al multiplicador de odds de pago entre las dos categorías, condicionado por el modelo». No es causal ni una justificación para actuar sobre grupos.

Para `age`, el IC anual es aproximadamente **[1,001; 1,039]**. Está cerca de 1: hay que distinguir una asociación estimada pequeña por año de su acumulación en varios años y de la incertidumbre de esa comparación.

En `credit.amo` puede aparecer `1.000` por redondeo. La unidad monetaria es pequeña y la visualización con tres decimales puede ocultar diferencias. **Un valor mostrado como 1 no demuestra un OR exactamente igual a 1.**

**Comprueba:** en cada fila identifica qué cambia —una unidad o una categoría—, cuál es la referencia y qué información añade el intervalo respecto al valor puntual.

## 4.7. Probabilidades, discriminación y calibración

### 4.7.1. Examinar probabilidades de entrenamiento

**Volvemos a la pregunta principal:** una vez estimados los coeficientes, ¿qué probabilidad de pago asigna el modelo a cada crédito? `predict(...)` combina las entradas con esos coeficientes y devuelve valores entre 0 y 1.

`pr` es una serie con **una probabilidad por fila de entrenamiento**. `pr.head()` solo muestra las cinco primeras; no limita el cálculo a cinco créditos. Al seleccionar `X[mod_glm1.model.exog_names]` mantenemos las columnas y el orden exactos que conoce el modelo.

Una probabilidad de 0,8 significa **80 % de pago estimado**, no 0,8 euros ni certeza de que ese caso se pague. El resultado real seguirá siendo 0 o 1.

A continuación localizamos el máximo y el mínimo para aprender a consultar filas. Es una inspección, no una evaluación completa. Dos casos favorables no representan a todos los créditos y no bastan para comprobar las probabilidades.

**Tres preguntas que separaremos:**

| Pregunta | Herramienta |
|---|---|
| ¿Asigna puntuaciones mayores a los pagos que a los impagos? | Discriminación: ROC y AUC |
| ¿Las probabilidades se parecen a las frecuencias reales? | Calibración |
| ¿Qué etiquetas acierta con una regla concreta? | Umbral y matriz de confusión |

Estas primeras comprobaciones usan datos que ya intervinieron en el ajuste; después repetiremos la evaluación con test.

In [ ]:
pr = mod_glm1.predict(X[mod_glm1.model.exog_names])  # Usa solo las entradas del modelo seleccionado, con el mismo orden.
display(pr.head())  # Conserva pr completa para gráficos y evaluación; muestra las primeras probabilidades.

#### Observación con la mayor probabilidad de pago

`pr.idxmax()` devuelve la **etiqueta de la fila** con mayor probabilidad predicha; no devuelve esa probabilidad. La guardamos en `pos_max` para consultar después dos cosas diferentes:

- `pr.loc[pos_max]`: la probabilidad estimada para esa fila.
- `datos.loc[pos_max, 'y']`: lo que realmente ocurrió.

`loc` selecciona por etiqueta, no necesariamente por posición. Aunque una etiqueta sea un número, no debes suponer que identifica la enésima fila de cualquier tabla.

**Qué debes observar:** incluso una probabilidad muy próxima a 1 no equivale a certeza. La consulta del resultado real sirve para comparar, no para calcular la predicción de esa fila.

In [ ]:
pos_max = pr.idxmax()  # Etiqueta de la fila, no necesariamente su posición.
pr.loc[pos_max]  # Probabilidad de pago estimada por el modelo.
datos.loc[pos_max, "y"]  # Resultado observado; no sustituye a la probabilidad.

#### Observación con la menor probabilidad de pago

Repetimos la consulta con `pr.idxmin()`, que identifica la etiqueta con la probabilidad **de pagar** más pequeña. La guardamos en `pos_min` y leemos su probabilidad y su respuesta real mediante `loc`.

**Atención al evento:** una probabilidad de pago de 0,04 corresponde a una probabilidad de impago de $1-0{,}04=0{,}96$. No cambia el significado de `pr`: sigue conteniendo probabilidad de pago para todas las filas.

**Comprueba:** distingue etiqueta, probabilidad y clase observada. Un crédito con probabilidad baja de pagar podría pagar; un único resultado no puede demostrar por sí solo si la cifra probabilística estaba bien estimada.

In [ ]:
pos_min = pr.idxmin()  # Etiqueta de la probabilidad de pago mínima.
pr.loc[pos_min]  # Probabilidad baja, no certeza de impago.
datos.loc[pos_min, "y"]  # Resultado observado en esta fila de entrenamiento.

### Lectura de los extremos

La probabilidad máxima es aproximadamente **0,999** y el resultado de ese caso fue `yes`; la mínima es aproximadamente **0,042** y su resultado fue `no`. En ambos extremos, el resultado coincide con la dirección de la puntuación.

**Lo que podemos decir:** el modelo asigna puntuaciones muy distintas a esos dos perfiles y, en estos dos casos, la clase observada es coherente con ellas.

**Lo que no podemos decir:** que todos los créditos se clasifican bien, que 0,999 significa certeza o que las probabilidades del conjunto estén calibradas. Para esas preguntas necesitamos más observaciones y herramientas específicas.

Piensa en una moneda que da cara con probabilidad 0,8: que una tirada concreta dé cruz no vuelve imposible esa probabilidad. Del mismo modo, un resultado inesperado no demuestra por sí solo que un modelo probabilístico sea incorrecto. Hay que estudiar patrones en conjuntos de casos.

El siguiente gráfico deja de mirar solo dos extremos y compara las distribuciones completas de las probabilidades.

#### Distribución de probabilidades según el pago observado

**Pregunta:** ¿el modelo suele asignar probabilidades de pago mayores a quienes realmente pagaron? Dibujamos una caja para los impagos observados y otra para los pagos observados.

**Cómo leer los ejes:**

- **Horizontal:** separa los casos por su resultado real, `No paga` y `Paga`.
- **Vertical:** probabilidad **predicha de pagar**, en ambos grupos. No es el importe ni la probabilidad de la clase que aparece debajo de cada caja.

La línea central es la mediana y la caja contiene el 50 % central. Los bigotes y puntos ayudan a ver dispersión y valores alejados según la regla del boxplot; un punto fuera de los bigotes no significa un dato erróneo.

Esperaríamos la caja de pagos más arriba que la de impagos. Si se solapan, habrá puntuaciones parecidas en ambas clases y puede haber errores con un umbral. El gráfico resume distribuciones: para contar errores exactos necesitaremos aplicar la regla y construir la matriz de confusión.

**No confundir separación con calibración:** el modelo puede ordenar bien los casos y, sin embargo, asignar probabilidades demasiado altas a todos ellos. Más adelante repetiremos estas cajas con test sin cambiar los coeficientes.

In [ ]:
from s4_graficos import cajas_probabilidad

cajas_probabilidad(pr, datos['y'], 'entrenamiento')  # Se reutiliza la misma función para test.

### 4.7.2. Curva ROC y AUC

**Pregunta:** ¿qué capacidad tiene el modelo para dar puntuaciones mayores a los pagos reales que a los impagos reales? Esto se llama **discriminación**. No requiere escoger todavía un único corte definitivo.

Para entender la ROC, imagina que clasificamos como pago los casos que superan un corte. Al bajar ese corte, más créditos pasan a ser predichos como pago: podemos detectar más pagos, pero también etiquetar más impagos como pagos. La ROC recorre esos compromisos.

El evento positivo sigue siendo **pagar (`yes`, 1)**. En sus ejes aparecen:

| Eje | Qué mide | Denominador |
|---|---|---|
| Vertical: sensibilidad | Pagos reales predichos como pago | Todos los pagos reales |
| Horizontal: $1-\text{especificidad}$ | Impagos reales predichos como pago, falsos positivos | Todos los impagos reales |

Un punto próximo a la esquina superior izquierda combina sensibilidad alta y pocos falsos positivos. La diagonal es una referencia de **discriminación aleatoria**, no una exactitud del 50 % para cualquier muestra.

El **AUC** es el área bajo la ROC. Se interpreta como la probabilidad de que, al elegir un pago y un impago reales al azar de la población de evaluación, el pago tenga mayor puntuación; los empates cuentan como medio acierto. En la muestra se estima mediante esas comparaciones. AUC = 0,5 corresponde a orden aleatorio y AUC = 1 a separación perfecta en la muestra.

**Tres diferencias importantes:**

- AUC no es el porcentaje de decisiones correctas con corte 0,5.
- AUC no verifica que las probabilidades coincidan con las frecuencias reales.
- AUC no incorpora directamente costes, equidad ni el umbral que convendría usar.

**Ejemplo para distinguir preguntas:** probabilidades 0,6 y 0,7 pueden ordenar dos casos igual que 0,8 y 0,9, pero no expresan las mismas probabilidades de pago. Mantener el orden no garantiza calibración.

**Primera lectura:** aprende los dos ejes y qué pregunta responde el AUC. No necesitas memorizar etiquetas universales de «bueno» o «malo»; su utilidad depende del problema.

In [ ]:
from s4_graficos import dibujar_roc

dibujar_roc(y, pr, 'Curva ROC — entrenamiento')  # Discriminación de las probabilidades ya calculadas.

In [ ]:
AUC_train = roc_auc_score(y, pr)  # Usa resultados reales y probabilidades de entrenamiento.
print(f'AUC en entrenamiento: {AUC_train:.3f}')  # No es el porcentaje de decisiones correctas.

### Lectura de la ROC de entrenamiento

Obtenemos **AUC ≈ 0,836**. La curva queda por encima de la diagonal y el modelo tiende a asignar puntuaciones mayores a los pagos reales que a los impagos reales.

**Interpretación aproximada:** en las comparaciones de un pago y un impago de esta muestra, el orden favorece al pago en alrededor del 83,6 % de las parejas, contando medio acierto en los empates. No afirma que el 83,6 % de los créditos esté correctamente clasificado con un corte concreto.

La llamada `roc_auc_score(y, pr)` necesita las **respuestas reales y las probabilidades**, no las etiquetas ya convertidas a pago/impago con un umbral. Si perdiéramos las puntuaciones, perderíamos información sobre su orden.

**Cautela:** estas filas se utilizaron para estimar y seleccionar el modelo. El resultado puede ser optimista y no anticipa por sí solo el AUC de casos nuevos.

**Comprueba antes de seguir:** explica por qué un AUC de 0,836 no permite afirmar que entre créditos con probabilidad 0,8 pagará exactamente el 80 %. Esa segunda pregunta pertenece a la calibración.

### 4.7.3. Calibración: probabilidad predicha frente a frecuencia real

**Pregunta:** cuando el modelo anuncia probabilidades cercanas al 80 %, ¿paga aproximadamente el 80 % de esos casos? A esa concordancia la llamamos **calibración**. Es una pregunta diferente de ordenar pagos e impagos.

La función agrupa las probabilidades en intervalos de **igual amplitud**: 0–0,1; 0,1–0,2; etc. No son los cuantiles de Hosmer–Lemeshow: cada tramo puede contener un número muy distinto de créditos y los vacíos no generan puntos.

Para cada grupo calcula:

| Coordenada | Cálculo | Ejemplo ficticio |
|---|---|---|
| Horizontal | Media de las probabilidades predichas | 0,80 |
| Vertical | Pagos observados / créditos del grupo | 6 / 10 = 0,60 |

Ese ejemplo daría el punto **(0,80; 0,60)**: el modelo esperaba más pagos de los que ocurrieron. La diagonal representa igualdad entre ambas cantidades.

- **Punto por debajo de la diagonal:** sobreestimación de la frecuencia de pago en el grupo.
- **Punto por encima:** subestimación.
- **Punto próximo:** concordancia aproximada, no prueba de una probabilidad individual exacta.

**Mira siempre `n`:** una proporción basada en tres créditos puede variar muchísimo si cambia un resultado. No debe interpretarse con la misma confianza que una basada en cientos. El gráfico une puntos para facilitar la lectura, pero las líneas no son un nuevo modelo ajustado.

La tabla devuelta se guarda en `calibracion_train`; el código no cambia las probabilidades ni reajusta coeficientes. Esta primera revisión es de entrenamiento. Repetiremos el mismo procedimiento en test.

**Comprueba:** un modelo puede discriminar razonablemente y sobreestimar todas las probabilidades. Por eso ROC/AUC y calibración se complementan, no se sustituyen.

In [ ]:
from s4_graficos import calibracion_modelo  # Muestra el gráfico y devuelve la tabla por intervalos.

In [ ]:
calibracion_train = calibracion_modelo(y, pr, intervalos=10, titulo="entrenamiento")  # Guarda la tabla mostrada; no reajusta.

### Lectura de la calibración de entrenamiento

En el intervalo de menor probabilidad hay **9 créditos**. La probabilidad media es **0,076**, pero pagó aproximadamente el **22,2 %**. El punto está por encima de la diagonal: el modelo subestima la frecuencia observada en ese pequeño grupo.

En el último intervalo hay **191 créditos**, probabilidad media **0,954** y proporción de pagos **0,948**. Es una concordancia bastante cercana en ese grupo, con un tamaño mucho mayor.

**Cómo comparar:** identifica primero el intervalo, después `n` y finalmente las dos proporciones. La discrepancia del grupo de nueve casos no debe interpretarse con la misma confianza que la del grupo de 191. Tampoco una buena concordancia en un extremo garantiza el resto del gráfico.

**Conclusión:** la concordancia cambia por grupos. Ni AUC ≈ 0,836 ni el p-valor alto de Hosmer–Lemeshow demostraban calibración perfecta; esta revisión añade información y sigue siendo en entrenamiento.

#### Micropráctica: cambiar los intervalos

1. Repite `calibracion_modelo(...)` con un número menor que diez; por ejemplo, considera cómo sería agrupar en cinco tramos.
2. Anticipa que habrá menos puntos y, al juntar intervalos, grupos generalmente mayores. Las proporciones pueden fluctuar menos, pero también pueden ocultarse discrepancias locales.
3. Compara los gráficos y los tamaños de las tablas. Identifica qué detalles desaparecen o cambian de dirección.
4. Explica qué no cambia: respuestas, coeficientes y probabilidades de cada crédito.

**No es una mejora automática del modelo:** cambiar los cortes modifica el resumen visual, no la calidad de las predicciones. No escojas solo el gráfico que parezca más cercano a la diagonal.

In [ ]:
try:
    calibracion_modelo(y, pr, intervalos=..., titulo="entrenamiento")  # Completa un número menor que los diez grupos del ejemplo.
except Exception as error:
    plt.close("all")  # Cierra una figura parcial si el número de intervalos no es válido.
    print("El microejercicio no está bien implementado todavía. Revisa el número de intervalos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## 4.8. De la probabilidad a una regla de decisión

### 4.8.1. Fijar el umbral antes de abrir test

**Pregunta:** si el modelo devuelve un número entre 0 y 1, ¿cómo obtenemos una etiqueta? Necesitamos una regla adicional, llamada **umbral** o corte. Para esta demostración fijamos **$c=0{,}5$**:

- Si $p>0{,}5$, predecimos pago.
- Si $p\le0{,}5$, predecimos impago. **El caso exactamente igual a 0,5 queda en impago**, según la convención del código.

La probabilidad y la etiqueta no son lo mismo: 0,51 y 0,99 producirían la misma clase con ese corte, aunque expresan estimaciones muy distintas.

### Un ejemplo pequeño, calculado a mano

Estos **seis casos ficticios** no forman parte del ajuste ni del test:

| Caso | Probabilidad de pago | Resultado real | Predicción con corte 0,5 | ¿Acierta? |
|---|---|---|---|---|
| A | 0,90 | Paga | Pago | Sí |
| B | 0,70 | No paga | Pago | No |
| C | 0,60 | Paga | Pago | Sí |
| D | 0,40 | Paga | Impago | No |
| E | 0,20 | No paga | Impago | Sí |
| F | 0,10 | No paga | Impago | Sí |

Hay **4 aciertos de 6**, pero todavía conviene separar los errores: un impago fue predicho como pago y un pago, como impago. El resultado real no participa en la regla: solo se utiliza después para comprobarla.

**Por qué importa el corte:** al subirlo exigimos más probabilidad para predecir pago. Normalmente se reconocen más impagos, pero se dejan escapar más pagos. Cambiarlo no modifica las probabilidades del modelo: cambia cómo las convertimos en clases.

La tabla del código compara 0,3, 0,5 y 0,7 **solo en entrenamiento** para entender ese compromiso. Guardamos `UMBRAL_FIJADO = 0.5` antes de abrir test; los controles no lo cambiarán.

**¿Es 0,5 el mejor corte?** No lo hemos demostrado. Con probabilidades bien calibradas y costes de error iguales puede ser una regla coherente, pero un uso real exige estudiar costes, calibración, equidad y restricciones legales. Para elegir empíricamente un corte haría falta validación interna, reservando un test independiente. La decisión `doy_credito` será solo una etiqueta ficticia de la práctica, no una recomendación real de préstamo.

In [ ]:
from s4_modelos import metricas_corte  # Cada métrica conserva su denominador explicado arriba.

metricas_cortes_train = pd.DataFrame([metricas_corte(y, pr, corte) for corte in (0.3, 0.5, 0.7)])
display(metricas_cortes_train.round(3))  # Comparación didáctica, no elección con test.
UMBRAL_FIJADO = 0.5  # Regla preespecificada; los controles no la modificarán.

### Lectura del compromiso entre errores

Al pasar del corte **0,3 al 0,7**, la sensibilidad baja aproximadamente de **97,5 % a 72,7 %** y la especificidad sube de **28,5 % a 77,1 %**.

**Traducción a palabras:**

- Con corte bajo, muchos casos reciben la etiqueta «pago»: se detecta gran parte de los pagos reales, pero también se etiquetan como pago bastantes impagos.
- Con corte alto, se exige más puntuación: se reconocen más impagos, pero algunos pagos reales quedan predichos como impago.

La exactitud combina ambas clases y no tiene por qué cambiar de forma monótona. Una mejora en una tasa puede acompañarse de un empeoramiento en otra. No existe un corte que resuma todos los objetivos sin definir sus prioridades.

**La fila de 0,5 es nuestra regla preespecificada**, no un óptimo demostrado. Las tres filas son de entrenamiento y pueden ser optimistas.

### 4.8.2. Explorar un corte sin cambiar la regla

El control siguiente conecta tres vistas de **las mismas probabilidades de entrenamiento**: matriz de confusión, métricas y punto sobre la ROC. Mueve el corte y observa qué celdas ganan o pierden casos.

**Qué permanece fijo:** los coeficientes, las probabilidades, la curva ROC completa, su AUC y `UMBRAL_FIJADO`. Solo cambian la clasificación exploratoria y el punto que corresponde a ese corte. No se consulta test.

Si no aparece el control, usa `mostrar_html(explorar_corte_entrenamiento(0.5))` con otro valor. Antes de mirar el resultado, anticipa qué tasa debería aumentar y cuál disminuir al subir el corte.

In [ ]:
from s4_graficos import explorar_corte_entrenamiento as _explorar_corte
from s4_controles import crear_control_corte

explorar_corte_entrenamiento = partial(_explorar_corte, y, pr, UMBRAL_FIJADO)  # Nunca utiliza test.
control_corte = crear_control_corte(explorar_corte_entrenamiento, umbral=UMBRAL_FIJADO)
display(control_corte)  # Solo cambia la vista, no el modelo ni la regla de evaluación.

#### Antes de convertir el modelo en una decisión real

**Que una predicción pueda calcularse no significa que deba utilizarse para decidir sobre una persona.** En esta sesión estudiamos un conjunto histórico para aprender estadística, no para recomendar una política de concesión.

El archivo contiene `status.sex` —incluye sexo—, `foreign` —condición de trabajador extranjero— y `age`. Pueden estar sujetos a restricciones legales según el lugar y el contexto, y relacionarse con diferencias entre grupos que el modelo no explica. **Una asociación no establece una causa ni justifica un trato desigual.**

Antes de cualquier uso fuera del aula habría que revisar, como mínimo:

1. **Datos:** procedencia, errores, actualidad, consentimiento cuando proceda y representatividad respecto a las personas afectadas.
2. **Pertinencia y licitud:** qué características pueden utilizarse legítimamente y con qué finalidad.
3. **Rendimiento por grupos:** calibración y tipos de error, con tamaños suficientes para estimarlos. Una métrica global puede ocultar diferencias importantes.
4. **Consecuencias:** costes de falsos positivos y negativos, posibilidad de explicación, revisión y supervisión humana.
5. **Seguimiento:** si el comportamiento cambia después de poner el sistema en uso.

Quitar una variable sensible tampoco demuestra equidad: otras columnas pueden actuar como sustitutos. Con subgrupos pequeños, las propias comparaciones de equidad tienen mucha incertidumbre.

**Debate breve:** nombra una comprobación estadística y otra no estadística necesarias antes de un despliegue. Ningún AUC, p-valor u OR de este cuaderno sustituye esas comprobaciones.

## Ejercicio 4.1 — Revisar el modelo con entrenamiento

**Objetivo:** construir una explicación completa utilizando resultados que ya has calculado: **selección → asociación e incertidumbre → diagnóstico**. No se trata de encontrar otro modelo ni de hacer nuevas elecciones con test.

Conserva `mod_glm0`, `mod_glm1`, referencias, probabilidades y umbral. Resolver estos apartados no es necesario para ejecutar después los ejemplos de test. Las celdas pendientes lo indican con un mensaje; sustituye el bloque de actividad por tu trabajo cuando estés preparado.

### 4.1.1. Resumir la selección

Construye una tabla con una fila para el modelo completo y otra para el seleccionado. Debe contener **AIC** y **número de variables originales**. Puedes añadir el número de coeficientes para mostrar por qué son cantidades diferentes.

**Guía de trabajo:**

1. Recupera los AIC con el atributo `.aic`.
2. Cuenta entradas con `len(predictores)` y `len(variables_seleccionadas)`.
3. Organiza los valores mediante `pd.DataFrame` y etiquetas claras.
4. Consulta `pasos_aic`: omite el paso inicial al buscar la primera retirada real.
5. Explica qué modelo tiene menor AIC y por qué la comparación usa la misma muestra.

**Tu conclusión debe distinguir:** menos variables, menos parámetros y menor AIC no son lo mismo, aunque puedan acompañarse. Un AIC menor no demuestra mejor calibración ni generalización. No sobrescribas los objetos originales para construir la tabla.

In [ ]:
try:
    # 4.1.1 — Resume la selección con entrenamiento, sin sobrescribir los modelos.
    # Pistas: usa .aic, len(predictores), len(variables_seleccionadas) y pasos_aic.
    # Construye una tabla con pd.DataFrame y comenta qué se simplificó.
    raise NotImplementedError("Completa la comparación de AIC y variables originales.")
except NotImplementedError as error:
    print(f"Ejercicio 4.1.1 pendiente: {error}")
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### 4.1.2. Interpretar un OR y su intervalo

**Objetivo:** comunicar una comparación categórica sin confundir odds y probabilidad.

Recupera el coeficiente `foreign=yes` del modelo seleccionado y los límites de `IC.loc['foreign=yes']`. Exponencia tanto el coeficiente como los límites. No necesitas volver a ejecutar la búsqueda del perfil para usar una fila ya calculada.

Tu respuesta debe incluir:

1. **Evento:** pagar, `y=yes`.
2. **Comparación:** `foreign=yes` frente a la referencia `foreign=no`.
3. **Estimación:** OR puntual y su IC del 95 %.
4. **Referencia del intervalo:** si incluye el valor **1**, que equivale a ausencia de cambio en odds.
5. **Condiciones y límites:** manteniendo constantes las demás entradas, con incertidumbre asociada a la muestra y a la selección previa.

**Ayuda para redactar:** usa «los odds de pagar se multiplican por ...» y evita «la probabilidad es ... %». Menciona que el grupo de referencia tiene solo 30 casos.

Cierra explicando por qué la asociación no demuestra causalidad ni justifica una decisión crediticia por pertenencia a un grupo. No uses esta comparación como una recomendación real de concesión.

In [ ]:
try:
    # 4.1.2 — Estimación e intervalo del OR para foreign=yes.
    # Pistas: np.exp(mod_glm1.params.loc[...]) y np.exp(IC.loc[...]).
    # Muestra el resultado e interpreta evento, referencia y valor 1.
    raise NotImplementedError("Completa el cálculo y la interpretación del odds ratio.")
except NotImplementedError as error:
    print(f"Ejercicio 4.1.2 pendiente: {error}")
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### 4.1.3. Diagnosticar y concluir

**Objetivo:** unir las piezas sin pedir a una sola métrica que responda todas las preguntas.

1. Ejecuta `hosmer_lemeshow(y, pr)` y consulta el elemento `'p-valor'` de la serie devuelta. Explica si rechazarías la adecuación agrupada al nivel 0,05, sin llamar al resultado «probabilidad de que el modelo sea correcto».
2. Revisa `calibracion_train`: elige dos grupos, informa sus tamaños y compara probabilidad media con proporción observada de pagos.
3. Recupera `AUC_train` y explica que mide orden entre clases, no exactitud del umbral ni concordancia de probabilidades.
4. Integra los resultados con la selección y el OR del apartado anterior.

**Estructura sugerida para tu conclusión:** muestra utilizada → modelo simplificado → asociación e intervalo → diagnóstico → limitación. Una limitación mínima es que todo este ejercicio usa entrenamiento, donde el ajuste y la selección pueden hacer las comprobaciones optimistas.

**Comprobación de comprensión:** ¿por qué puede coexistir un p-valor alto de Hosmer–Lemeshow con desviaciones en algún intervalo de calibración? Porque cambian las agrupaciones, se resume información y ninguna herramienta demuestra por sí sola calibración perfecta.

No consultes test para modificar el modelo, las referencias o el umbral.

In [ ]:
try:
    # 4.1.3 — Integra contraste y calibración del entrenamiento.
    # Pistas: hosmer_lemeshow(y, pr), calibracion_train y AUC_train.
    # Explica qué informa cada resultado y redacta tu conclusión.
    raise NotImplementedError("Completa el diagnóstico y la conclusión en contexto.")
except NotImplementedError as error:
    print(f"Ejercicio 4.1.3 pendiente: {error}")
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## 4.9. Evaluar con datos de test

### 4.9.1. Leer y codificar sin volver a ajustar

**Pregunta:** ¿cómo funciona el protocolo completo con créditos que no se usaron para aprenderlo? Hasta aquí hemos fijado los coeficientes, las variables y el corte 0,5 usando solo entrenamiento o decisiones previas. Ahora abrimos el conjunto reservado.

Piensa en entrenamiento como el material con el que se aprende y en test como una prueba que no debe haberse utilizado para corregir previamente el método. Si cambiamos el modelo al ver sus resultados, este test dejaría de ser una prueba independiente de la revisión.

**Qué hace la celda, paso a paso:**

1. Lee los créditos de test y comprueba ausentes y valores válidos de la respuesta.
2. Compara los niveles categóricos con los conocidos en entrenamiento. Una categoría nueva produce un error; no se convierte silenciosamente en la referencia.
3. Aplica **el mismo orden de categorías**, para mantener los mismos indicadores y las mismas referencias.
4. Crea `X_test` con las columnas del diseño de entrenamiento y las ordena igual. `reindex` alinea nombres; no vuelve a estimar coeficientes.
5. Crea `y_test` con pago = 1, únicamente para las comprobaciones posteriores.

**Diferencia entre los objetos:** `test` es la tabla reservada; `X_test`, sus entradas codificadas; `y_test`, sus resultados reales. La respuesta no entra en `X_test`. Que `test` la contenga no significa que `predict` la utilice.

**Qué no ocurre aquí:** no se llama a `.fit()` con test, no se seleccionan variables y no se cambia `UMBRAL_FIJADO`. Una predicción debe corresponder a cada fila, utilizando la misma interpretación de las columnas.

Primero evaluaremos probabilidades —discriminación y calibración— y después etiquetas con la regla ya fijada. No uses ninguna de esas salidas para volver atrás y optimizar con este mismo test.

In [ ]:
test = pd.read_csv(DATA_PATH / "bank0_test.txt", sep=";")  # Lee únicamente los créditos reservados para evaluar.
if test.isna().any().any() or not test['y'].isin(['no', 'yes']).all():
    raise ValueError('Revisa ausentes o categorías de respuesta inesperadas en test.')
for columna in datos.select_dtypes(include="category"):
    desconocidas = set(test[columna].dropna()) - set(datos[columna].cat.categories)  # Contrasta con niveles de entrenamiento.
    if desconocidas:
        raise ValueError(f"Categorías nuevas en {columna}: {desconocidas}")
    test[columna] = pd.Categorical(test[columna], categories=datos[columna].cat.categories)  # Mismo orden y referencia.

X_test = pd.get_dummies(test[predictores], drop_first=True, prefix_sep="=", dtype=float)  # No incorpora y.
X_test = sm.add_constant(X_test, has_constant="add").reindex(columns=X.columns, fill_value=0)  # Mismo diseño y orden.
y_test = (test["y"] == "yes").astype(int)  # Pagar sigue siendo el evento positivo.
print(f'Créditos de test: {len(test)}')
display(test['y'].value_counts().rename('Resultados observados'))

### Lectura de los datos de test

Hay **313 créditos reservados**: **227 pagos y 86 impagos**. La proporción de pagos es $227/313\approx72{,}5\%$, algo mayor que el 68,9 % observado en entrenamiento.

**Una referencia útil:** predecir siempre pago acertaría el 72,5 % en este test, pero no detectaría ningún impago. Así entenderemos después por qué la exactitud aislada puede parecer aceptable aunque una clase se reconozca mal.

Los 313 casos no se añaden a los 687 de entrenamiento para ajustar ahora el modelo. Se mantienen separados y sus respuestas se utilizan para evaluar. La diferencia de proporciones entre muestras no identifica por sí sola su causa ni demuestra un cambio de población.

### 4.9.2. Predecir probabilidades y revisar discriminación

La llamada `mod_glm1.predict(...)` aplica los **coeficientes ya aprendidos** a `X_test` y guarda una probabilidad de pago por fila en `pr_test`. Los nombres de las columnas garantizan que cada coeficiente multiplique la entrada que le corresponde.

Las comprobaciones posteriores exigen:

- **Alineación:** cada probabilidad conserva la etiqueta de su fila de test.
- **Valores finitos:** no hay ausentes ni infinitos.
- **Dominio válido:** todas las probabilidades están entre 0 y 1.

`pr_test.head()` enseña solo las primeras, pero `pr_test` conserva todas. Los boxplots, la ROC y `roc_auc_score(y_test, pr_test)` usan los resultados observados para evaluar esas probabilidades; no alteran su cálculo.

**Antes de ejecutar:** anticipa qué mirarás en las cajas y en la ROC. La separación entre clases informa sobre discriminación, no sobre calibración ni sobre el número exacto de errores con el corte 0,5.

In [ ]:
pr_test = mod_glm1.predict(X_test[mod_glm1.model.exog_names])  # Coeficientes de entrenamiento; entradas en el mismo orden.
if not pr_test.index.equals(test.index) or not np.isfinite(pr_test).all() or not pr_test.between(0, 1).all():
    raise ValueError('Revisa la alineación y el dominio de las probabilidades de test.')
print(f'Probabilidades obtenidas: {len(pr_test)}')
display(pr_test.head())  # Muestra una parte; pr_test completa se conserva para evaluar.

In [ ]:
cajas_probabilidad(pr_test, test["y"], "test")  # Mismo gráfico que en entrenamiento, ahora con casos reservados.

In [ ]:
dibujar_roc(y_test, pr_test, "Curva ROC — test")  # Evalúa el orden de las probabilidades sin seleccionar otro corte.

In [ ]:
AUC_test = roc_auc_score(y_test, pr_test)  # Discriminación en los créditos reservados.
display(pd.Series({'Entrenamiento': AUC_train, 'Test': AUC_test}, name='AUC').round(3))  # No son porcentajes de acierto.

### Lectura de la discriminación en test

Las probabilidades suelen ser mayores para los pagos reales, pero las cajas **se solapan**. Hay pagos con puntuación baja e impagos con puntuación alta: el modelo no separa perfectamente ambas clases.

El AUC pasa de aproximadamente **0,836 en entrenamiento a 0,749 en test**. En esta muestra reservada, el orden sigue siendo mejor que la referencia aleatoria, aunque discrimina menos que en las filas utilizadas para ajustar y seleccionar.

**Cómo redactarlo:** «La discriminación observada disminuye fuera del entrenamiento». No equivale a «el 74,9 % de los créditos está bien clasificado» ni permite atribuir con certeza la diferencia a **sobreajuste**: aprender particularidades o ruido del entrenamiento que no se mantienen con casos nuevos. Los AUC dependen de sus muestras; también podrían influir diferencias entre los casos y variación muestral.

No hemos calculado aquí un intervalo de confianza para esa diferencia. Por tanto, describimos los valores observados, no una prueba formal de que sus rendimientos poblacionales sean distintos.

**Siguiente pregunta:** aunque el modelo ordene razonablemente, ¿acierta la magnitud de las probabilidades? Para eso repetiremos la calibración. El modelo y el umbral permanecen fijos.

### 4.9.3. Calibración fuera de entrenamiento

**Pregunta:** ¿las probabilidades aprendidas con entrenamiento se corresponden con frecuencias reales de pago en casos nuevos? Repetimos el gráfico con **diez intervalos de anchura fija**, usando el mismo procedimiento que antes.

La llamada `calibracion_modelo(y_test, pr_test, ...)` reúne información de dos orígenes: la probabilidad procede del modelo entrenado y el resultado real, del test. La función compara, dibuja y devuelve una tabla; **no ajusta ni recalibra el modelo**.

Para leer cada fila y cada punto sigue siempre este orden:

1. Identifica el intervalo de probabilidades.
2. Revisa cuántos créditos contiene (`n`).
3. Compara la probabilidad media horizontal con la proporción real de pagos vertical.
4. Describe si queda por encima o por debajo de la diagonal y evita convertir una diferencia muestral en una certeza sobre futuros clientes.

Los tramos tienen el mismo ancho que en entrenamiento, pero no necesariamente los mismos tamaños ni las mismas medias predichas. Los vacíos no aparecen como puntos y los muy pequeños tienen proporciones inestables.

**Comprueba:** puede mantenerse el AUC y empeorar la calibración, porque una transformación que conserve el orden de las puntuaciones puede cambiar sus valores. Por eso necesitamos ambas lecturas.

Si este diagnóstico motivara revisar o recalibrar el modelo, la revisión debería aprenderse con datos adecuados de desarrollo y evaluarse con otros datos nuevos. No podemos corregir con este test y seguir tratándolo como evaluación independiente de esa corrección.

In [ ]:
calibracion_test = calibracion_modelo(y_test, pr_test, intervalos=10, titulo="test")  # Misma función y probabilidades ya fijadas.

### Lectura de la calibración en test

El primer intervalo contiene solo **3 créditos**. Su probabilidad media es aproximadamente **0,075**, pero pagaron **2 de 3**, el **66,7 %**. El punto queda muy por encima de la diagonal: la frecuencia observada supera mucho la anunciada por el modelo para ese grupo.

**Por qué hay que mirar el tamaño:** con tres casos, cambiar un resultado mueve la proporción en unos 33,3 puntos porcentuales. La discrepancia merece atención, pero la cifra de ese grupo es muy incierta y no debe generalizarse sin más.

En el último intervalo hay **96 créditos**, probabilidad media **0,953** y proporción de pagos **0,917**. El punto está por debajo de la diagonal: el modelo sobreestima algo la frecuencia observada de pago. La diferencia es de unos **3,6 puntos porcentuales**, no un recuento de 3,6 clientes.

En las zonas intermedias aparecen discrepancias en ambos sentidos. El AUC no podía garantizar concordancia de probabilidades: solo resumía el orden entre clases.

**Conclusión adecuada:** las probabilidades y las frecuencias agrupadas no coinciden exactamente en este test; hay que valorar tamaños, variabilidad muestral y uso previsto. No hemos demostrado que cada probabilidad individual sea correcta o incorrecta.

Conservamos los coeficientes y las probabilidades. El siguiente paso no corrige el modelo: aplica el umbral ya fijado y cuenta sus decisiones.

## 4.10. Evaluar decisiones con la regla fijada

### 4.10.1. Sensibilidad, especificidad y exactitud

**Pregunta:** ¿qué tipos de casos reconoce la clasificación y cuáles confunde? Aplicamos el mismo corte **0,5** a entrenamiento y test: pago si $p>0{,}5$ e impago si $p\le0{,}5$.

Para nombrar los resultados, recuerda que **positivo significa pago** en este cuaderno:

| Nombre | Qué ocurrió | Qué predijo la regla |
|---|---|---|
| Verdadero positivo, TP | Pagó | Pago |
| Falso negativo, FN | Pagó | Impago |
| Verdadero negativo, TN | No pagó | Impago |
| Falso positivo, FP | No pagó | Pago |

«Verdadero» significa acierto y «falso», error; «positivo» no significa aquí impago. Si cambiáramos el evento de interés tendríamos que reinterpretar todas las tasas.

| Métrica | Pregunta | Cálculo |
|---|---|---|
| Sensibilidad | De todos los pagos reales, ¿cuántos detectamos? | $TP/(TP+FN)$ |
| Especificidad | De todos los impagos reales, ¿cuántos reconocemos? | $TN/(TN+FP)$ |
| Exactitud | De todos los créditos, ¿cuántos clasificamos bien? | $(TP+TN)/n$ |

**Recupera los seis casos ficticios del apartado 4.8:** había dos pagos detectados de tres pagos reales, dos impagos reconocidos de tres impagos y cuatro aciertos de seis casos. Las tres tasas dan 2/3 en ese ejemplo, pero **sus denominadores responden a preguntas diferentes** y no tienen por qué coincidir en otros datos.

La función `metricas_corte` calcula estas tasas con las respuestas y probabilidades suministradas. La tabla las coloca juntas para comparar entrenamiento y test sin cambiar modelo ni corte.

**No leas solo exactitud:** una clase muy frecuente puede producir muchos aciertos aunque la otra se reconozca mal. La aceptabilidad de los errores depende de sus consecuencias; no hemos asignado costes monetarios.

**Regla de evaluación:** no cambies ahora el corte para maximizar el resultado del test. Eso convertiría sus respuestas en parte del proceso de elección.

In [ ]:
comparacion_decisiones = pd.DataFrame({  # El modelo y el corte no cambian entre columnas.
    "Entrenamiento": metricas_corte(y, pr, UMBRAL_FIJADO),
    "Test": metricas_corte(y_test, pr_test, UMBRAL_FIJADO),
})
display(comparacion_decisiones.round(3))  # Sensibilidad, especificidad y exactitud usan denominadores diferentes.

### Lectura de las decisiones

Con el corte fijo **0,5**, las tasas cambian de la siguiente manera:

| Métrica | Entrenamiento | Test | Lectura en test |
|---|---|---|---|
| Sensibilidad | 87,7 % | 85,5 % | Se reconocen aproximadamente 85,5 de cada 100 pagos reales |
| Especificidad | 53,7 % | 50,0 % | Se reconoce la mitad de los impagos reales |

Estas frases describen proporciones de la muestra; no significan que podamos identificar de antemano qué persona concreta será un error.

La **exactitud en test es aproximadamente 75,7 %**. Si siempre predijéramos pago, obtendríamos 72,5 %. Hay una mejora observada del total, pero su interpretación es incompleta sin las tasas por clase: reconocer solo la mitad de los impagos podría ser importante según el uso y sus costes.

**Distinción de escalas:** la tabla del código muestra tasas entre 0 y 1; por ejemplo, 0,855 equivale a 85,5 %. No significa 0,855 %.

**Qué no se ha demostrado:** que la diferencia frente al referente sea estadísticamente concluyente, que los costes sean aceptables o que el rendimiento sea igual en todos los grupos. No hemos calculado aquí intervalos para esas tasas.

Ahora reconstruiremos los conteos de la matriz para comprobar exactamente los numeradores y denominadores de estos porcentajes.

### 4.10.2. Matriz de confusión y porcentajes por decisión

**Una matriz de confusión es una tabla de doble entrada:** cruza lo que predijo la regla con lo que ocurrió. No contiene probabilidades, sino recuentos de casos.

La columna `doy_credito` es la **decisión ficticia**: `si` si la probabilidad de pago supera 0,5 y `no` en caso contrario. Sus etiquetas no son las mismas que las de `y`, que utiliza `no` y `yes`.

**Convención de este cuaderno:**

| Filas: decisión / columnas: realidad | No pagó (`no`) | Pagó (`yes`) |
|---|---|---|
| Predice impago (`no`) | TN, acierto | FN, pago no detectado |
| Predice pago (`si`) | FP, impago predicho como pago | TP, acierto |

Algunos materiales ponen realidad en filas y predicción en columnas: sería la tabla transpuesta. Antes de calcular nada, comprueba los rótulos, no memorices solo una posición.

**Ejemplo de los seis casos ficticios:** la fila `no` contendría 2 impagos y 1 pago; la fila `si`, 1 impago y 2 pagos. Los aciertos serían 2 + 2. Así puedes reconstruir la tabla a mano a partir de las seis filas del apartado 4.8.

**Qué hace el código real:** `np.where(...)` convierte la comparación con el umbral en etiquetas; `pd.crosstab(...)` cruza decisión y resultado; `reindex(...)` fija el orden y conserva una celda aunque su recuento sea cero. Primero se muestran conteos, después porcentajes.

Para los porcentajes usamos **el total de cada fila**. La operación `sum(axis=1)` suma a lo largo de las columnas de cada fila. Dividir cada celda por ese total permite preguntar «dentro de esta decisión, ¿qué ocurrió?».

**No es lo mismo que sensibilidad o especificidad:** esas tasas condicionan al resultado real, por lo que sus denominadores son los totales de las columnas en esta tabla. Si una fila está vacía, su porcentaje es indefinido y se muestra como ausente, no como 0 %.

La tabla describe la práctica; no recomienda conceder o denegar préstamos reales.

In [ ]:
test["doy_credito"] = np.where(pr_test > UMBRAL_FIJADO, "si", "no")  # Aplica la regla original, sin usar el pago real.
matriz_test = pd.crosstab(test["doy_credito"], test["y"]).reindex(
    index=['no', 'si'], columns=['no', 'yes'], fill_value=0)  # Filas: decisión; columnas: realidad.
display(matriz_test)

In [ ]:
porcentajes_test = matriz_test.div(matriz_test.sum(axis=1).replace(0, np.nan), axis=0).mul(100)  # Cada fila es el denominador.
display(porcentajes_test.round(1))  # Una fila vacía tendría porcentaje indefinido, no 0 %.

### Lectura de la matriz de confusión

En test, la matriz contiene:

| Decisión / resultado real | No paga | Paga | Total de la decisión |
|---|---|---|---|
| `no` | 43 | 33 | 76 |
| `si` | 43 | 194 | 237 |
| Total real | 86 | 227 | 313 |

**Comprueba las tasas anteriores con estos conteos:** sensibilidad $194/227\approx85{,}5\%$; especificidad $43/86=50\%$; exactitud $(43+194)/313\approx75{,}7\%$.

Ahora toma los **43 impagos predichos como pago**. Podemos dividirlos por dos totales distintos:

| Pregunta | Cálculo | Resultado aproximado |
|---|---|---|
| Entre las decisiones `si`, ¿qué proporción acabó en impago? | $43/237$ | 18,1 % |
| Entre todos los impagos reales, ¿qué proporción se predijo como pago? | $43/86$ | 50,0 % |

El segundo porcentaje es la **tasa de falsos positivos**, igual a $1-\text{especificidad}$. El primero es la proporción de errores dentro de las decisiones positivas. El numerador coincide, pero **cada denominador define una pregunta diferente**.

Ninguno de esos porcentajes es un coste monetario: no incluye importes, pérdidas, recuperaciones ni otras consecuencias.

#### Micropráctica: interpretar una fila

1. Reutiliza `pd.crosstab(..., normalize='index')`, que convierte cada fila en proporciones que suman 1.
2. Multiplica por 100 para expresarlas como porcentajes.
3. Completa la etiqueta de la decisión de conceder y consulta la columna de impagos, `'no'`.
4. Escribe la frase «de los créditos clasificados como ..., ... % ...» e indica explícitamente el denominador.

**Comprueba:** la normalización por filas responde a una pregunta distinta de dividir por todos los impagos. No llames a ambos resultados «tasa de impago» sin aclarar entre qué casos.

In [ ]:
try:
    proporciones_test = (100 * pd.crosstab(test["doy_credito"], test["y"], normalize="index")).round(1)
    clase_concedido = ...  # Usa la etiqueta de fila que aparece en la matriz de confusión.
    porcentaje_impago_concedido = proporciones_test.loc[clase_concedido, "no"]  # Extrae la proporción de impago dentro de esa fila.
    print(f"Impagos entre créditos concedidos: {porcentaje_impago_concedido:.1f} %")
except Exception as error:
    plt.close("all")
    print("El microejercicio no está bien implementado todavía. Revisa la etiqueta de fila y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### 4.10.3. Ubicar la regla en la ROC de test

**Objetivo:** conectar la matriz de confusión con un punto del gráfico que ya conocemos. La ROC completa resume las mismas probabilidades de test para distintos cortes; el marcador identificado en la leyenda como **«Corte fijado = 0.50»** señala únicamente nuestra regla fija 0,5.

Sus coordenadas se obtienen de los conteos anteriores:

$$x=1-\text{especificidad}=43/86=0{,}50,\qquad y=\text{sensibilidad}=194/227\approx0{,}855.$$

**Cómo leerlo:** en esta muestra, la regla detecta alrededor del 85,5 % de los pagos reales, mientras clasifica como pago el 50 % de los impagos reales. No son porcentajes dentro de las decisiones `si`.

La función repite conteos, porcentajes y métricas para que compruebes los denominadores. No estima nuevos coeficientes ni modifica `pr_test`; la curva y su AUC permanecen iguales. La llamada sin argumentos funciona porque `partial` ya dejó fijados datos y corte.

**Comprueba:** localiza 0,50 en el eje horizontal y 0,855 en el vertical. Explica por qué el 18,1 % de impagos entre decisiones de conceder no es la coordenada horizontal.

No utilices este gráfico para elegir ahora otro corte. Si quisieras revisar la regla, define previamente el criterio y un protocolo de validación con datos de desarrollo, y reserva una evaluación nueva independiente. La figura no autoriza una política real de concesión.

In [ ]:
from s4_graficos import evaluar_umbral_fijado as _evaluar_umbral

evaluar_umbral_fijado = partial(_evaluar_umbral, y_test, pr_test, UMBRAL_FIJADO)  # Modelo y corte ya fijados.

In [ ]:
evaluar_umbral_fijado()  # Repite y relaciona las métricas ya obtenidas; no reajusta ni elige un nuevo corte.

## Ejercicio 4.2 — Evaluar y comunicar resultados

**Objetivo:** construir un informe que distinga **probabilidades, decisiones y denominadores**. Utiliza el modelo y las predicciones ya calculados; no busques otros cortes para que el test parezca mejorar.

Conserva `mod_glm1`, `pr_test` y `UMBRAL_FIJADO`. Usa nombres nuevos para tus tablas y termina cada apartado con una interpretación, no solo una salida numérica.

### 4.2.1. Reconstruir decisiones y métricas

**Guía de trabajo:**

1. Compara `pr_test` con el umbral usando la misma regla del cuaderno: **`>`**, no `>=`.
2. Cruza esa clase predicha con `y_test` mediante `pd.crosstab(...)`. Decide y escribe qué representan filas y columnas; si usas booleanos, `True` significa predice pago y 1 significa pago real.
3. Identifica TP, FN, TN y FP en tu propia tabla antes de dividir.
4. Calcula sensibilidad, especificidad y exactitud con sus fórmulas, o compruébalas con `metricas_corte(...)`.
5. Compara con entrenamiento usando exactamente el mismo corte.

**Comprobaciones:** el total de la matriz debe ser 313; los denominadores de sensibilidad y especificidad son los pagos e impagos reales, respectivamente. Distingue tasas entre 0 y 1 de porcentajes entre 0 y 100.

Redacta qué clase reconoce mejor la regla en este test y una limitación. No presentes el AUC como porcentaje de aciertos ni cambies el umbral por los resultados obtenidos.

In [ ]:
try:
    # 4.2.1 — Mismo modelo y umbral, sin nuevas elecciones con test.
    # Pistas: pd.crosstab(...), metricas_corte(y_test, pr_test, UMBRAL_FIJADO).
    # Compara con entrenamiento y explica los denominadores.
    raise NotImplementedError("Completa la matriz y las métricas del umbral fijado.")
except NotImplementedError as error:
    print(f"Ejercicio 4.2.1 pendiente: {error}")
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### 4.2.2. Comparar dos porcentajes de impago

**Objetivo:** demostrar que un numerador no basta para interpretar un porcentaje. Usa la matriz del ejemplo o la que acabas de reconstruir, respetando su orientación.

Calcula por separado:

1. **Impagos entre decisiones `si`:** numerador = impagos en la fila de conceder; denominador = total de esa fila.
2. **Tasa de falsos positivos:** mismo numerador; denominador = total de impagos reales, que en la matriz del cuaderno es la columna `'no'`.

Puedes recuperar el numerador con `matriz_test.loc['si', 'no']`. Para los denominadores, revisa qué suma una fila y qué suma una columna. No mezcles la matriz de conteos con una tabla que ya esté normalizada.

**Presentación requerida:** una pequeña tabla con nombre de la medida, numerador, denominador y porcentaje. Conserva precisión completa en los cálculos y redondea solo al mostrar.

Después redacta una frase de cada medida y comprueba que la segunda coincide con $100\,(1-\text{especificidad})$. Explica por qué son diferentes sin que haya contradicción: condicionan a grupos distintos.

**Límite:** no son costes monetarios. Para hablar de pérdidas económicas necesitaríamos datos y supuestos adicionales que esta práctica no aporta.

In [ ]:
try:
    # 4.2.2 — Numerador igual, denominadores distintos.
    # Pistas: matriz_test.loc['si', 'no'], total de fila 'si' y total de columna 'no'.
    # Calcula ambos porcentajes con precisión completa y redondea solo al mostrar.
    raise NotImplementedError("Completa los porcentajes y sus interpretaciones.")
except NotImplementedError as error:
    print(f"Ejercicio 4.2.2 pendiente: {error}")
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### 4.2.3. Comunicar probabilidades y limitaciones

**Objetivo:** redactar un informe que pueda entender una persona que no ha ejecutado el cuaderno. No es una lista de métricas sin contexto.

Organiza un texto breve en cinco partes:

1. **Datos y protocolo:** tamaños de entrenamiento y test, evento pagar y separación entre desarrollo y evaluación.
2. **Modelo y regla:** modelo seleccionado con entrenamiento y corte 0,5 fijado antes de abrir test. Indica que no se reajustaron con sus resultados.
3. **Probabilidades:** compara `AUC_train` y `AUC_test`; después elige uno o dos grupos de `calibracion_train` y `calibracion_test` para explicar qué aporta la calibración. Informa sus tamaños.
4. **Decisiones:** resume sensibilidad, especificidad y exactitud con sus denominadores. Si incluyes impagos entre decisiones de conceder, aclara que no es la tasa de falsos positivos.
5. **Incertidumbre y uso:** menciona variabilidad muestral, grupos pequeños y que no hemos estimado intervalos para todas las métricas. Añade una precaución sobre calidad de datos, equidad o licitud.

**Para evaluar una revisión:** usaríamos validación interna o datos de desarrollo distintos para aprender el nuevo modelo o umbral y, después, **nuevos datos de test independientes** para evaluarlo. Este test ya visto no puede certificar de manera independiente una revisión elegida por sus resultados.

**Revisión final de tu texto:** elimina frases como «probabilidad garantizada», «AUC = exactitud» o «el OR demuestra que una característica causa impago». La conclusión no debe recomendar decisiones reales de préstamo.

In [ ]:
try:
    # 4.2.3 — Resume discriminación, calibración y decisiones con las mismas reglas.
    # Pistas: AUC_train, AUC_test, calibracion_train, calibracion_test y comparacion_decisiones.
    # Redacta una conclusión, una limitación y un protocolo para evaluar posibles revisiones.
    raise NotImplementedError("Completa la comparación y el informe contextual.")
except NotImplementedError as error:
    print(f"Ejercicio 4.2.3 pendiente: {error}")
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## Cierre de la sesión

Hemos pasado de una respuesta binaria a **probabilidades de pago** y, mediante una regla explícita, a etiquetas de una decisión ficticia. La selección por AIC reduce **20 entradas originales a 11** con los **687 créditos de entrenamiento**. Sus indicadores no son nuevas características originales ni más clientes.

### Qué resultado describe cada herramienta

| Herramienta | Pregunta que responde | Qué no demuestra por sí sola |
|---|---|---|
| Ajuste logístico | ¿Cómo relaciona el modelo las entradas con la probabilidad? | Causalidad o validez de todos los supuestos |
| AIC | ¿Qué candidato ofrece mejor balance ajuste–complejidad? | Mejor rendimiento futuro |
| OR e IC | ¿Qué asociación ajustada se estima y con qué precisión? | Cambio fijo de probabilidad |
| ROC y AUC | ¿Cómo ordena pagos frente a impagos? | Calibración o exactitud de un corte |
| Calibración | ¿Se parecen probabilidades y frecuencias agrupadas? | Corrección de cada probabilidad individual |
| Matriz y tasas | ¿Qué acierta y confunde una regla concreta? | Costes aceptables o equidad |

### Qué ocurrió en el test reservado

En **313 créditos**, el modelo obtiene aproximadamente **AUC = 0,749**, **sensibilidad = 85,5 %**, **especificidad = 50,0 %** y **exactitud = 75,7 %**, con corte fijo **0,5**. Los **impagos entre decisiones de conceder son 18,1 %**: ese denominador son las decisiones `si`, no todos los impagos reales.

Estas cifras describen una muestra reservada, no garantizan precisión individual ni rendimiento en otra población. La calibración mostró discrepancias y algunos grupos tenían pocos casos. No hemos evaluado costes económicos ni demostrado un uso justo o legal de todas las entradas.

### Autoevaluación antes de terminar

Debes poder explicar, con palabras y un ejemplo:

1. La diferencia entre resultado 0/1, probabilidad y clase predicha.
2. Por qué hay más coeficientes que variables originales y qué es una referencia.
3. Por qué OR = 2 no duplica necesariamente una probabilidad.
4. Por qué un p-valor alto no prueba que el modelo sea correcto.
5. Por qué discriminación y calibración son preguntas distintas.
6. Cómo calcular sensibilidad, especificidad y exactitud desde los conteos.
7. Por qué 43/237 y 43/86 pueden ser correctos a la vez.
8. Por qué no se modifica el modelo ni el umbral para mejorar el test ya visto.

Si algún punto no está claro, vuelve al ejemplo pequeño del apartado correspondiente antes de intentar memorizar fórmulas. Para comprobar reproducibilidad, reinicia el kernel y ejecuta en orden. Los ejercicios conservan sus huecos para que practiques.

**Uso responsable:** esta sesión enseña estadística, no recomienda conceder o denegar préstamos. Revisar modelo o regla después de usar este test requiere un protocolo de desarrollo adecuado y otros datos independientes para la nueva evaluación.

### Documentación de las herramientas

No necesitas leer toda la documentación para completar la primera vuelta. Utilízala para resolver una duda concreta sobre los argumentos o las salidas de una función:

- [Modelos lineales generalizados en statsmodels](https://www.statsmodels.org/stable/glm.html): qué hacen `GLM`, la familia binomial y el ajuste.
- [Curva ROC en scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html): diferencia entre respuestas reales, puntuaciones y cortes.
- [Calibración en scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.calibration_curve.html): idea de comparar probabilidad media y frecuencia observada por grupos. En este cuaderno la agrupación y el gráfico están implementados en el script auxiliar.

**Para revisar el código de apoyo:** modelos y métricas están en [s4_modelos.py](s4_modelos.py), gráficos en [s4_graficos.py](s4_graficos.py) y controles en [s4_controles.py](s4_controles.py). Todos reciben sus datos explícitamente; el cuaderno fija qué tabla y qué modelo utiliza cada ejemplo.

**Cómo plantear una duda:** anota el apartado, qué pregunta intentas responder, la llamada ejecutada y la salida o mensaje exacto. Distingue un ejercicio aún sin completar de un error al ejecutar el ejemplo. Eso permite localizar si la dificultad está en el concepto, en una etiqueta de columna o en el estado del kernel.